# Модель водохозяйственной системы бассейна реки Зарафшан
*Курс, основанный на моделировании*
**Авторы:** Доктор Тобиас Зигфрид, Николас Лазаро, Доктор Беатрис Марти (гидросолюшнс ГмбХ)  
**Лицензия:** CC BY 4.0.

---

<div style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #2780e3; background-color: #eaf3fc;"> 
 <strong> &#x1F4DD; Примечание. Цели обучения </strong> <br> <br>

После изучения пособия вы сможете:

1. **Описать** топологию и компоненты узлово-рёберной модели водохозяйственной системы: источники, водохранилища, узлы водопотребления, распределения, транзита и выхода из системы.
2. **Прочитать** таблицу водного баланса и объяснить физический смысл притока, потребления, потерь, изменения запаса и невязки баланса.
3. **Сравнить** стратегии распределения по показателям, учитывающим потребность: результативности водообеспечения и доле избыточной подачи. Объяснить, почему простой показатель эффективности системы может вводить в заблуждение.
4. **Проанализировать** влияние потерь при транспортировке воды по каналам и определить приоритетные вложения в инфраструктуру по потерям на отдельных участках сети.
5. **Использовать** TaqSim и пакет `zarafshan_taqsim` для создания, запуска и анализа сценариев водохозяйственной системы в Python.
6. **Объяснить** архитектуру TaqSim на основе событий: как цикл расчёта, эксплуатационные стратегии и предоставление параметров оптимизатору обеспечивают воспроизводимый анализ и автоматическую оптимизацию (см. [Раздел 9](#sec-appendix-taqsim)).
7. **Понять** основы многокритериальной оптимизации: как целевые функции, фронты Парето и алгоритм NSGA-II связывают результаты моделирования с анализом компромиссов.

</div> 
 <div style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #e6a817; background-color: #fdf6e3;"> 
 <strong> &#x26A0; &#xFE0F; Предупреждение: Предварительные требования </strong> <br> <br>

Для работы с пособием необходимы следующие знания:

- **Математика**: базовая алгебра, соотношения и пропорции, преобразование единиц измерения между расходами и временными объемами (от m<sup>3</sup>/s до m<sup>3</sup>/day, а также ежемесячные итоги, где это необходимо).
- **Область**: Вводные концепции гидрологии (речной сток, суммарное испарение, водный баланс). Если вы незнакомы, см. [Гролл и др. (2015)](https://doi.org/10.1007/s12665-013-2988-5), чтобы узнать о контексте гидрологии Центральной Азии.
- **Вычислительные**: знание Python (функции, словари, генераторы списков), знакомство с pandas DataFrames и matplotlib/построение графиков.
- **Концептуальный**: Ориентированные графы (узлы и ребра). Никакого предварительного опыта моделирования или оптимизации водохозяйственных систем не требуется.

</div> 
 <details style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #2780e3; background-color: #eaf3fc;"> 
 <summary> <strong> &#x1F4DD; Примечание. Единицы измерения и условные обозначения </strong> </summary>

| Символ/Единица измерения | Значение |
|:-----------------------------------|:-----------------------------------|
| м³/с | Расход (кубических метров в секунду) |
| м³/день | Объем за дневной временной шаг модели = расход $\times$ 86 400 с |
| м³/месяц | Ежемесячная общая сумма исходных данных или ежемесячная совокупность ежедневных результатов |
| мм³ | Мегакубические метры ( $10^6$ м³) — используются для определения емкости резервуаров и годовых объемов компонентов |
| Бм³ | Миллиард кубических метров ( $10^9$ м³) — используется для общего годового притока и объемов на уровне системы |
|  $S_d$ | Секунды в день = 86 400 с |
|  $S_m$ | Среднее количество секунд в месяц $\approx 2{,}630{,}016$ с (= 30,44 дня $\times$ 86 400 с/день) |
| \% Приток | Компонент водного баланса, выраженный в процентах от общего притока системы |

Исполняемая модель выполняется с **ежедневным интервалом** ( `TIMESTEPS = 2191` , охватывающим 2017–2022 гг.). Приток реки закачивается в виде ежедневно наблюдаемых рядов. Несколько исходных наборов данных сохраняются ежемесячно и при необходимости тиражируются или дезагрегируются для получения ежедневных входных данных модели. В таблицах для удобства чтения обычно указываются годовые объемы в миллиметрах или миллиметрах.

</details> 
 <a id="sec-introduction"> </a>

# Введение
 <a id="sec-zarafshan-basin"> </a>

## Бассейн реки Зарафшан
Река Зарафшан (также называемая Зеравшан) — одна из наиболее значительных трансграничных рек в Центральной Азии, протекающая через Таджикистан и Узбекистан ([Гролл и др., 2015 г.](https://doi.org/10.1007/s12665-013-2988-5)). Его название на персидском языке означает «Рассыпание золота», имея в виду исторические месторождения золота на его берегах. Протянувшись примерно на 877 км от ледника Зарафшан в Таджикистане (2775 метров над уровнем моря) до его исчезновения в пустыне Кызылкум, река поддерживает площадь бассейна примерно 17 700 km<sup>2</sup> и обеспечивает орошаемое земледелие на протяжении более 2500 лет ([Каримов и др., 1995 г.](https://www.fao.org/4/v9529e/v9529e06.htm)). Древние города Самарканд и Бухара — оба объекта Всемирного наследия ЮНЕСКО — были построены вдоль его берегов и по сей день зависят от его вод.

Зеравшан имеет *ниво-ледниковый* гидрологический режим, то есть его сток питается преимущественно за счет таяния снегов и ледникового льда, а не осадков (Шульц, 1965). В верхнем бассейне горный водосборный бассейн Матча (4550 км²) содержит 118 ледников, покрывающих 270,7 км² льда, на которые приходится примерно 22% годового стока по всему бассейну и до 37% в пиковые летние месяцы с июля по сентябрь. Осадки в качестве прямого источника стока незначительны, составляя всего лишь 1% годового стока на гидрометрическом посту Дупули в Таджикистане — главной гидрометрической станции на входе в орошаемые низменности, расположенной в нескольких десятках километров выше места впадения реки в Узбекистан. Это ледниковое и нивальное доминирование приводит к резкому сезонному гидрографу: только июль обеспечивает более 25% годового стока, а март — до начала сезона таяния снегов — составляет менее 2%.

В Дупули средний расход составляет примерно 155–160 м³/с. Примечательно, что межгодовая изменчивость реки Зерафшан является одной из самых низких среди всех рек Центральной Азии: коэффициент вариации составляет всего C <sub> v </sub> ≈ 0,08 (Шульц, 1965). Эта стабильность — следствие режима стока, буферизованного ледниками, и средней высоты водосбора примерно 3000 метров над уровнем моря — означает, что даже сильные засухи приводят к сравнительно умеренному сокращению стока: во время засухи в Центральной Азии 1917 года расход Зерафшана упал всего на 10%, тогда как соседняя река Чирчик снизилась на 40%. Бассейн питается за счет Гиссарского хребта, на южных склонах которого выпадает более 2000 мм осадков в год, хотя в Самарканде в низменности выпадает только 323 мм/год, при этом 39% приходится на декабрь-январь и всего 1,5% на июнь-август. Ниже точек ирригационного водозабора в современном Узбекистане естественный гидрограф постепенно трансформируется: водозаборы срезают летний пик, в то время как возвратные оросительные потоки создают искусственный зимний пик, создавая характерный двухпиковый годовой цикл на станциях ниже по течению (Шульц, 1965).

Современный Зарафшан сталкивается с множеством проблем, которые делают его идеальным примером для моделирования водохозяйственной системы. Весь сток реки уже распределён; на равнинных равнинах (площадь около 11 800 км²) потери стока на орошение, фильтрация и испарение превышают скорость формирования стока в 1,5 раза, а в засушливые годы пользователи нижнего течения сталкиваются с серьезным дефицитом (Shultz, 1965; [Кулматов и др., 2013.](https://doi.org/10.4236/jwarp.2013.51003)). Таджикистан контролирует верховья, в то время как Узбекистан испытывает большую часть сельскохозяйственного спроса, создавая трансграничную напряженность ([Гролл и др., 2015 г.](https://doi.org/10.1007/s12665-013-2988-5)). Каналы и водохранилища советской эпохи требуют модернизации, а отступление ледников в условиях изменения климата ставит под угрозу долгосрочную доступность воды, особенно с учетом того, что ледниковый вклад в летний сток составляет 22-37%. Сельское хозяйство, городское снабжение, промышленность и экологические потоки конкурируют за один и тот же ограниченный ресурс.

<figure id="fig-basin-map" style="margin:1em 0; text-align:center;"> 
 <img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/zarafshan_river_basin_map.png" alt="GIS map of the Zarafshan River Basin." style="max-width:100%; height:auto;"> 
 <figcaption style="font-style:italic; color:#555; margin-top:4px;"> <b> Рисунок 1 </b>: ГИС-карта нижнего течения реки Зарафшан. Река и ее рукава показаны синим цветом, течение которых направлено справа налево. Основные искусственные сооружения управления перенаправлением и хранением выделены оранжевыми кружками; два водохранилища расположены в средней части реки. Источник: Джованоли (2025). </figcaption> 
 </figure>

[Рисунок 1](#fig-basin-map) показывает нижнюю часть бассейна реки Зарафшан, которая служит территорией нашего исследования. На карте обозначены границы исследуемой территории, реки, каналы, два водохранилища (Каттакурганское и Акдарьинское), орошаемые площади и Навоийская ТЭЦ. Ключевые гидросооружения — водозаборные сооружения, которые разделяют речной поток между конкурирующими рукавами — выделены оранжевым цветом. Орошаемые низменности разделены на административные ирригационные районы, охватывающие несколько областей (Самаркандская, Навоийская, Кашкадарьинская, Джизакская); Подробная карта района с цветовой кодировкой представлена ​​в [Рисунок 4](#fig-irrigation-districts).

<figure id="fig-canal-network" style="margin:1em 0; text-align:center;"> 
 <img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/zarafshan_magistral_canal_system.png" alt="Magistral canal system of the Zarafshan River Basin." style="max-width:100%; height:auto;"> 
 <figcaption style="font-style:italic; color:#555; margin-top:4px;"> <b> Рисунок 2 </b>: Система магистральных каналов бассейна реки Зарафшан с указанием проектной пропускной способности (Q, м³/с) в ключевых узлах инфраструктуры. Пронумерованные узлы: (1) Раватхоза, (2) Ак-Карадарья, (3) Дамхоза, (4) Каттакурганское водохранилище, (5) Нарпай, (6) Акдарьинское водохранилище, (7) Карманинский гидроузел и (8) межбассейновые перепускные пункты в Джизак и Кашкадарью (серые вставки). Источник: Министерство водного хозяйства Узбекистана через Джованоли (2025). </figcaption> 
 </figure>

[Рисунок 2](#fig-canal-network) показывает инженерную схему системы магистральных каналов, предоставленную Министерством водного хозяйства Узбекистана [MoWR; Джованоли (2025). Магистральные (главные) каналы окрашены в ярко-синий цвет, а вторичные и третичные распределительные каналы видны как более тонкие ветви. Пронумерованные узлы 1–7 соответствуют ключевым компонентам инфраструктуры, использованным в имитационной модели: семи водозаборным сооружениям HydroWorks и двум водохранилищам. Два узла, обозначенные цифрой 8, обозначают точки межбассейнового переброса в Джизакскую и Кашкадарьинскую области, показанные вставками серого цвета.

Следуя потоку с востока на запад (справа налево на рисунке), вода поступает в **Гидроузлу Раватхоза**(узел 1), которая является первым крупным водозаборным пунктом на реке Зарафшан. Здесь сток разделяется на основное русло реки, продолжающееся до Ак-Карадарьи (Q = 885 м³/с проектная пропускная способность), Мирзапайский и Даргомский каналы (Q = 125 м³/с каждый) и Джизакский межбассейновый переброс (Q = 45 м³/с). На**Ак-Карадарьинском гидроузле**(узел 2) основная река далее разделяется на рукав Дамхоза (Q = 550 м³/с), канал Аккарадарья (Q = 120 м³/с) и приток в**Акдарьинское водохранилище**(узел 6, Q = 230 м³/с).**Дамхозинский гидроузел**(узел 3) распределяет сток по р. Карадарья (Q = 350 м3/с), каналу Нарпай (Q = 80 м3/с), каналу Мианкалтосс (Q = 100 м3/с) и в**Каттакурганское водохранилище**(узел 4, Q = 100 м3/с). Ниже по течению**Нарпайский гидроузел**(узел 5) и два водохранилища обеспечивают среднесистемное регулирование до того, как поток вновь сходится на**Карманский гидроузел** (узел 7), который распределяет воду в Навоийскую область (Q = 400 м³/с) и соответствующую Навоийскую теплоэлектростанцию ​​([Рисунок 3](#fig-navoi-intake)).

<figure id="fig-navoi-intake" style="margin:1em 0; text-align:center;"> 
 <img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/IMG_6666.jpeg" alt="Water intake structure at the Karmana Hydroworks." style="max-width:100%; height:auto;"> 
 <figcaption style="font-style:italic; color:#555; margin-top:4px;"> <b> Рисунок 3 </b> : Водозаборное сооружение на Карманинском гидроузле, отводящее поток на Навоийскую теплоэлектростанцию. Вдалеке видны градирни электростанции. Источник: Тобиас Зигфрид, Hydrosolutions GmbH. </figcaption> 
 </figure>

В этом учебнике исследуется центральный вопрос: **какое значение имеет правило распределения в этих точках водозабора?** Одна только потребность в сельскохозяйственном орошении составляет примерно 24% от общего притока системы, но бассейн также должен обслуживать значительные обязательства по межбассейновым переброскам в Бухару (через Навои), Джизак и Кашкадарью, что вместе увеличивает общие обязательства примерно до 53% притока. После учета потерь при транспортировке воды через сеть преимущественно земляных каналов, система действительно ограничена водой. Задача двоякая: *распределение* — доставка нужного количества воды в нужное место в нужное время — и *эффективность* — минимизация потерь в сети каналов, которая теряет значительную часть потока на фильтрацию и испарение. Мы строим модель системы ( [Раздел 3](#sec-system-model) ), изучаем исторические модели спроса и предложения ( [Раздел 4](#sec-historical-data) ), затем сравниваем три все более сложные стратегии распределения ( [Раздел 5](#sec-baseline-simulations) ), включая роль потерь при транспортировке воды по каналу ( [Раздел 5.4.2](#sec-canal-losses) ).

<a id="sec-taqsim"> </a>

## Моделирование водохозяйственной системы с помощью TaqSim
TaqSim — это среда моделирования водохозяйственной системы на основе событий ( [Лазаро и Зигфрид, 2024 г.](https://github.com/hydrosolutions/taqsim) ) (подробную архитектурную информацию см. в [Раздел 9](#sec-appendix-taqsim)). В отличие от традиционных моделей, которые напрямую отслеживают переменные состояния, TaqSim записывает **события** – неизменяемые записи движения воды (вода получена, выпущена, сохранена, потеряна), которые можно воспроизвести и проанализировать после моделирования. Эта архитектура позволяет восстанавливать любое состояние системы на любом временном этапе, анализировать закономерности в различных сценариях и создавать собственные метрики на основе трассировок событий без повторного запуска моделирования.

Основные абстракции TaqSim просты. **WaterSystem** – это ориентированный граф **узлов**, соединенных **ребрами**. Узлы представляют собой физические компоненты — водохранилища, потребности в орошении, точки водозабора, — тогда как ребра представляют собой инфраструктуру транспортировки воды (каналы, трубы) с ограничениями пропускной способности и свойствами потерь. **Стратегии** — это правила, которые управляют поведением узла: сколько воды выпускает резервуар и как узел отвода распределяет поток между своими нисходящими ветвями (различие между операционными стратегиями и физическими моделями объясняется в [Раздел 9.5](#sec-taqsim-strategies)). Выбор стратегии разделения является центральной переменной в этом учебнике: мы сравниваем равномерное разделение (равные доли для всех отраслей), статическое разделение, пропорциональное спросу (одно фиксированное соотношение, основанное на долгосрочном спросе), и изменяющееся во времени правило разделения с учетом спроса, сезонная структура которого применяется к часам ежедневного моделирования.

После моделирования журнал событий TaqSim позволяет составить комплексный **водный баланс**— учет сохранения массы, который отслеживает каждый кубический метр от притока до потребления, потерь при транспортировке воды, изменений в хранилище и оттока из терминала. Водный баланс является основным аналитическим инструментом в этом учебнике, который используется для сравнения стратегий и диагностики того, где вода добывается и теряется. TaqSim также предоставляет**цели**— функции, которые оценивают работу системы для**многоцелевой оптимизации**(одновременная минимизация нескольких конкурирующих целей, таких как сельскохозяйственный дефицит и разливы водохранилищ). В сопутствующем учебнике по оптимизации используется NSGA-II, эволюционный алгоритм, для поиска**оптимальных по Парето** решений — конфигураций, в которых ни одна цель не может быть улучшена без ухудшения другой (см. [Раздел 12.1](#sec-taqsim-objectives) для системы целей и [Раздел 14](#sec-glossary) для определений).

<details style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #2780e3; background-color: #eaf3fc;"> 
 <summary> <strong> &#x1F4DD; Примечание. Как работает источник событий </strong> </summary>

Чтобы конкретизировать концепцию, основанную на событиях, рассмотрим одно ребро сети, представляющее канал, несущий 50 m<sup>3</sup>/s за один ежедневный временной интервал в $t = 150$ в 2017 году. TaqSim записывает три события на этом ребре:

| Событие | Сумма | Значение |
|:----|---------------------------------------:|:--------------------------|
|  `WaterReceived(t=150)` | 4.32 Mm<sup>3</sup> | Поток, входящий в канал в тот день |
|  `WaterLost(t=150, reason='seepage')` | 0,32 Mm<sup>3</sup> | Потери фильтрации через русло канала |
|  `WaterDelivered(t=150)` | 4,00 Mm<sup>3</sup> | Поток, выходящий из канала к следующему узлу |

Никакая переменная текущего состояния не обновляется — каждое событие представляет собой неизменяемую запись. После моделирования `compute_water_balance()` суммирует все события `WaterLost` (по причине) и все события `WaterReceived` (по типу узла), чтобы восстановить полный баланс системы. Вот почему вы можете сравнивать стратегии, просто повторно запрашивая журнал событий с различными функциями агрегирования без повторного запуска моделирования.

</details> 
TaqSim объединяет несколько типов функциональных узлов с транспортными сегментами для моделирования водохозяйственной системы (подробное описание обучения представлено в [Раздел 9.2](#sec-taqsim-nodes)):

| Тип узла | Поведение |
|:------------|:-----------------------------------------------------|
| Источник | Генерирует воду (приток рек, сток) |
| Хранение | Хранит воду с правилом сброса (водоемы, озера) |
| Спрос | Потребляет воду, фиксирует дефицит (орошение, города) |
| Сплиттер | Делит поток между несколькими нисходящими путями |
| Проход | Подает воду с ограничением пропускной способности (каналы, турбины) |
| Выходной узел | Конечный узел выхода воды из системы |

Пакет Python `zarafshan_taqsim` реализует эту системную модель. Его шесть модулей — для загрузки данных, операционных стратегий, сборки сети, анализа водного баланса, визуализации и целей оптимизации — задокументированы в [Раздел 13](#sec-module-reference). Базовая архитектура платформы TaqSim документирована в [Раздел 9](#sec-appendix-taqsim).

<details style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #3fb618; background-color: #edf7ea;"> 
 <summary> <strong> &#x1F4A1; Совет: Упражнение А: Концептуальное представление — эскиз минимальной водохозяйственной системы </strong> </summary>

**Задание:** На бумаге (или в инструменте для рисования) нарисуйте минимальную систему водоснабжения ровно с четырьмя узлами: один источник, одно хранилище (водохранилище), одно требование (ирригационный район) и один сток. Нарисуйте направленные рёбра, показывающие, как вода течет от источника к стоку.

1. Сколько ребер имеет ваша система? Существует ли более одной допустимой топологии?
2. Если Источник генерирует 100 m<sup>3</sup>/s, а Потребление требует 60 m<sup>3</sup>/s с эффективностью 80 %, сколько воды забирает узел Потребления? Сколько достигает Раковины?
3. Если резервуар полон, что происходит с поступающей водой, которую невозможно хранить?

**Подсказка:** Узел спроса снимает `requirement / efficiency` = 60/0,8 = 75 m<sup>3</sup>/s . Остальные 25 m<sup>3</sup>/s проходят в Sink. Переполнение резервуара регистрируется как событие ущерба.

**Ожидаемый результат:** Ваша система должна иметь 3 ребра (Источник→Хранилище→Спрос→Приемник — одна допустимая топология; Источник→Спрос→Хранилище→Приемник – другая). Спрос снимает 75 m<sup>3</sup>/s и проходит 25 m<sup>3</sup>/s. В зависимости от того, где вы разместите резервуар, существует несколько допустимых топологий.

</details> 
 <div style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #d63384; background-color: #fce4f0;"> 
 <strong> &#x2757; Важно: Основные выводы — Введение </strong> <br> <br>

- Река Зарафшан полностью распределена: общие обязательства (орошение + межбассейновые переброски) достигают ~53% притока, оставляя ограниченную прибыль после потерь в канале.
- TaqSim — это платформа моделирования на основе событий: она записывает неизменяемые события (получение, потеря, накопление воды), что обеспечивает гибкий анализ после моделирования.
- Система водоснабжения моделируется как ориентированный граф функциональных узлов (Источник, Хранение, Спрос, Разделитель, Сквозной, Sink), соединенных транспортными сегментами, которые перемещают воду и могут нести потери.
- Центральный вопрос: **насколько важно правило разделения в точках перенаправления?**

</div>

<p> <img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/hydrosolutions_logo.png" alt="hydrosolutions" width="220"> </p> 
 <p> <b> Hydrosolutions GmbH </b> · Venusstrasse 29, 8050 Цюрих, Швейцария · <a href="https://www.hydrosolutions.ch"> www.гидроsolutions.ch </a> · hs@гидроsolutions.ch · +41 43 535 05 80 </p>

In [ ]:
# @title Подготовка: сначала выполните эту ячейку
# Подготовка: сначала выполните эту ячейку. В Colab это занимает около минуты.
# Загружаем папку курса, устанавливаем TaqSim и указываем Python путь к коду курса.
import os
import sys
import subprocess

if not os.path.exists("src/zarafshan_taqsim"):  # Пока мы вне папки курса (обычная ситуация в Colab).
    if not os.path.exists("zarafshan-taqsim-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/hydrosolutions/zarafshan-taqsim-course.git"], check=True)
    os.chdir("zarafshan-taqsim-course")
# Если ячейка уже выполнялась, в среде может остаться старая копия: обновляем папку и
# удаляем ранее загруженные модули курса из памяти, чтобы импортировать актуальный код.
if os.path.exists(".git"):
    subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
for name in [m for m in sys.modules if m == "zarafshan_taqsim" or m.startswith("zarafshan_taqsim.")]:
    del sys.modules[name]
try:
    import taqsim  # noqa: F401  (при работе на своём компьютере пакет уже установлен)
except ImportError:
    %pip install -q deap ctrl-freak pymoo
    %pip install -q --no-deps "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"
sys.path[:0] = [os.path.abspath("src")]
print('Готово. Рабочая папка:', os.getcwd())


In [ ]:
# @title Моделирование водохозяйственной системы с TaqSim · код
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from zarafshan_taqsim import (
    DEFAULT_DISTRICTS,
    build_baseline_overview_table,
    create_zrb_system,
    figures,
    tables,
    get_node_counts,
    load_canal_properties,
    load_demand,
    load_demand_config,
    load_inflow,
    load_min_flow,
    load_precipitation,
    load_reservoir_config,
    load_runoff_config,
    plot_supply_demand,
    plot_baseline_strategy_comparison,
    plot_canal_loss_breakdown,
    visualize_network,
    plot_static_ratio_network,
    compute_water_balance,
    plot_system_dashboard,
    plot_top_reach_losses,
    print_annual_water_balance,
    print_district_comparison,
    print_district_summary,
    print_powerplant_comparison,
    print_powerplant_stats,
    print_water_balance_comparison,
    apply_demand_proportional_splitting,
    apply_static_demand_splitting,
)

# Настройки графиков
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
figures.reset(start=4)
tables.reset()

# Настройки модели
TIMESTEPS = 2191  # Суточный расчёт за 2017–2022 годы (включая 29 февраля 2020 года)
SEED = 42

<a id="sec-system-model"> </a>

# Модель системы
 <a id="sec-network-topology"> </a>

## Топология сети
Топология сети имитационной модели ориентирована на два основных источника данных: имеющуюся инфраструктуру сети каналов и административное деление бассейна на ирригационные районы. Схема магистрального канала предоставлена ​​Министерством водного хозяйства Узбекистана [МиВР; Джованоли (2025) определяет физическую основу системы — основные каналы, их разветвленную структуру, проектную пропускную способность и расположение гидротехнических сооружений, контролирующих распределение стока (см. [Рисунок 2](#fig-canal-network)). Вторым ориентиром является разделение орошаемых низменностей на административные ирригационные округа, как показано в [Рисунок 4](#fig-irrigation-districts).

<figure id="fig-irrigation-districts" style="margin:1em 0; text-align:center;"> 
 <img src="https://hydrosolutions.github.io/zarafshan-taqsim-course/images/maps_irrigation_districts_zrb_USAID.jpg" alt="Irrigation map of the Zarafshan Valley." style="max-width:100%; height:auto;"> 
 <figcaption style="font-style:italic; color:#555; margin-top:4px;"> <b> Рисунок 4 </b> : Ирригационная карта Зерафшанской долины (Ирригационная карта Зерафшанской Долины), составленная Проектом USAID по управлению природными ресурсами в Центральной Азии (NRMP). Ирригационные районы имеют цветовую маркировку по областям: Навоийский (зеленый), Самарканд (розовый), Кашкадарьинская (желтая), Джизакская (персиковая) и Ленинабад, ныне Согдийская область, Таджикистан (фиолетовый). Синими линиями отмечена сеть магистральных каналов, голубыми линиями обозначены дренажные коллекторы, а оранжевыми символами отмечены гидротехнические сооружения и насосные станции. На карте показано, как ирригационные районы, такие как Даргом, Мирзапай, Нарпай, Мианкалт-Тосс, Аккарадарья и Иски-Ангар, обслуживаются выделенными водоотводящими каналами, ответвляющимися от главного ствола. Источник: USAID (без даты), воспроизведено через Giovanoli (2025). </figcaption> 
 </figure>

Границы районов с цветовой кодировкой в ​​[Рисунок 4](#fig-irrigation-districts) показывают, как вода Зеравшана разделена между конкурирующими административными регионами. Каждый крупный ирригационный район соответствует зоне обслуживания, питаемой одним или несколькими магистральными каналами, ответвляющимися в точке водозабора гидроузла. В имитационной модели эти районы объединены в шесть узлов водопотребления, каждый из которых представляет совокупную потребность в воде — потери при транспортировке, потери при внесении на поля и эвапотранспирацию сельскохозяйственных культур — всего ирригационного района. Агрегирование на уровне района является осознанным выбором моделирования: оно соответствует пространственному масштабу, в котором принимаются решения о распределении на гидроузлах и в котором данные о спросе доступны с помощью дистанционного зондирования ([Рагеттли и др., 2025 г.](https://doi.org/10.1016/j.ejrh.2025.102185)). На карте также видны обязательства по межбассейновой перекачке воды в Джизак (северо-восток) и Кашкадарью (юг), которые обслуживаются выделенными каналами, ответвляющимися от основной сети и смоделированными в моделировании как узлы-приемники.

Объединив эти два источника данных, модель ZRB представляет водохозяйственную систему как ориентированный граф функциональных узлов управления водными ресурсами, соединенных транспортными сегментами. Вода поступает в семь узлов источников: один представляет собой реку Зарафшан на гидропосте Раватхоза, а шесть представляют собой местный сток, вызванный осадками, на водосборном бассейне каждого ирригационного района. Затем он проходит через разветвленную сеть точек отвода HydroWorks (узлы разделения), двух резервуаров (узлов хранения), шести сельскохозяйственных ирригационных районов (узлов водопотребления), одной тепловой электростанции (узла PassThrough) и трех оконечных узлов стока. Между этими функциональными узлами модель включает транспортные сегменты, которые представляют каналы и участки рек и учитывают потери при транспортировке.

In [ ]:
# @title Топология сети · код
system = create_zrb_system()
system.validate()
fig, ax = visualize_network(system, show_reaches=False)
figures.caption(
    fig,
    'Топология сети Зарафшана: участки Reach сведены к направленным рёбрам. Синие узлы — источники воды, зелёные — водохранилища, оранжевые — распределительные гидроузлы HydroWorks, красные — ирригационные районы, фиолетовый — Навоийская ТЭС, серые — конечные выходные узлы.',
    label="network",
)

[Рисунок 5](#fig-network) отображает топологию бассейна с узлами Reach, свернутыми в направленные ребра, а цвета узлов указывают тип. Топология отражает основную структуру инфраструктуры распределения воды ZRB (см. реальную схему канала в [Рисунок 2](#fig-canal-network) и карту ирригационного района в [Рисунок 4](#fig-irrigation-districts)): единый источник, который постепенно разветвляется на подсети, обслуживающие разные регионы, с двумя водохранилищами, обеспечивающими сезонную буферизацию. Каждое ребро имеет расчетные физические свойства — тип облицовки (земляная, частичная или бетонная), состояние канала, длина и ширина — которые определяют потери при транспортировке во время транспортировки (происхождение данных см. в [Раздел 4.2](#sec-canal-properties)). Эти потери являются основным фактором работы системы и анализируются в [Раздел 5.4.2](#sec-canal-losses).

На рисунке показано, что к каждому узлу спроса прикреплен узел источника, который представляет воду из местных источников. Потоки оцениваются в зависимости от размера участвующих водосборов.

<a id="sec-node-categories"> </a>

## Категории узлов
 [Таблица 1](#tbl-node-counts) суммирует распределение узлов по типам.

In [ ]:
# @title Категории узлов · код
counts = get_node_counts(system)
counts_df = pd.DataFrame([
    {'Тип узла': k, 'Количество': v, 'Назначение': p}
    for k, v, p in [
        ("Source", counts.get("Source", 0), 'Речной приток из Таджикистана (1) + сток от осадков (6)'),
        ("Storage", counts.get("Storage", 0), 'Каттакурган (667 млн м³) и Акдарья (106 млн м³)'),
        ("Splitter", counts.get("Splitter", 0), 'Распределительные гидроузлы HydroWorks'),
        ("Demand", counts.get("Demand", 0), 'Сельскохозяйственные ирригационные районы'),
        ("PassThrough", counts.get("PassThrough", 0), 'Навоийская ТЭС (вода для охлаждения)'),
        ("Sink", counts.get("Sink", 0), 'Конечные узлы (Джизак, Кашкадарья, Навои)'),
    ]
])
tables.caption(
    counts_df.style.hide(axis="index"),
    'Состав узлов водохозяйственной модели Зарафшана.',
    label="node-inventory",
)

<a id="tbl-node-counts"> </a>

*Таблица 1: Распределение типов основных узлов в сети ZRB.*

<a id="sec-reservoirs"></a>

## Два водохранилища

В бассейне Зарафшана два крупных водохранилища обеспечивают сезонное регулирование стока ([Ragettli et al., 2025](https://doi.org/10.1016/j.ejrh.2025.102185); [CAWater-Info, 2024](https://cawater-info.net/bk/1-1-1-1-3-uz_e.htm)). **Каттакурганское водохранилище** (объём 667 млн м³, построено в 1965 году) служит для многолетнего регулирования и смягчения засух. **Акдарьинское водохранилище** (объём 106 млн м³, построено в 1967 году) обеспечивает сезонное регулирование; в частности, оно способствует надёжному водоснабжению Навоийской ТЭС ниже по течению ([Rakhmatullaev & Huneau, 2011](https://doi.org/10.1007/s12665-010-0711-3)). В модели мёртвый объём обоих водохранилищ принят равным 10% полного объёма. Это вода ниже отметки водовыпускных сооружений, которую нельзя выпустить.

Правило управления по уровню наполнения (SLOP) выделяет четыре зоны: мёртвый объём (менее 10%; попуск невозможен), буферную зону (сокращённый попуск для сохранения запаса), зону нормальной работы (целевой расход попуска) и противопаводковую зону (увеличенный попуск избытка воды, чтобы предотвратить неконтролируемый перелив). Целевой расход `vr` по умолчанию определяется эвристическим правилом месячного водообмена: $v_r = (C \times 0.20) / S_m$, где $C$ — полный объём водохранилища, а $S_m$ — среднее число секунд в месяце. Расход дополнительно ограничивается суммарной пропускной способностью водовыпускной инфраструктуры ниже по течению. Таким образом, за средний месяц выпускается примерно 20% полного объёма. Такое простое эксплуатационное правило не позволяет небольшим водохранилищам опорожняться слишком быстро и обеспечивает достаточный водообмен в крупных, соблюдая ограничения водовыпусков. Параметры попуска циклически повторяются по месяцам, но сам расчёт выполняется с суточным шагом.

<details style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #2780e3; background-color: #eaf3fc;">
<summary><strong>Примечание: зоны SLOP</strong></summary>

Четыре зоны непосредственно связаны с наполнением водохранилища:

```text
Запас воды (%)
100 ┬──────────── Полный объём ────────────────────
    │ ПРОТИВОПАВОДКОВАЯ ЗОНА: попуск > vr
 90 ├──────────────────────────────────────────────
    │ НОРМАЛЬНАЯ РАБОТА: попуск = vr
 30 ├──────────────────────────────────────────────
    │ БУФЕРНАЯ ЗОНА: попуск < vr
 10 ├──────────────────────────────────────────────
    │ МЁРТВЫЙ ОБЪЁМ: попуск = 0 (невозможен)
  0 └──────────────────────────────────────────────
```

**Пример расчёта (Каттакурган):** полный объём $C$ = 667 млн м³. Мёртвый объём — 67 млн м³ (10%). Пусть текущий запас равен 400 млн м³ (60% полного объёма). Это **зона нормальной работы** (между 30% и 90%), поэтому попуск равен целевому расходу $v_r$. По умолчанию $v_r = (667 \times 10^6 \times 0.20) / 2{,}630{,}016 \approx 50.7$ м³/с, что соответствует примерно 133 млн м³ за средний месяц. На каждом суточном шаге этот расход переводится в суточный объём. При снижении запаса до 130 млн м³ (20%) водохранилище входит в **буферную зону** и сокращает попуск для сохранения воды. Ниже 67 млн м³ оно находится в **зоне мёртвого объёма** и попуск прекращается.

</details>


In [ ]:
# @title Два водохранилища · код
reservoir_configs = load_reservoir_config()
rows = []
for res in reservoir_configs:
    capacity = res.capacity_m3 / 1e6
    dead = res.dead_storage_m3 / 1e6
    initial = res.initial_storage_m3 / 1e6
    active = capacity - dead
    rows.append({
        'Водохранилище': res.name.replace("RES_", ""),
        'Полный объём (млн м³)': f"{capacity:,.0f}",
        'Мёртвый объём (млн м³)': f"{dead:,.0f} ({100*dead/capacity:.0f}%)",
        'Полезный объём (млн м³)': f"{active:,.0f} ({100*active/capacity:.0f}%)",
        'Начальный запас (млн м³)': f"{initial:,.0f} ({100*initial/capacity:.0f}%)",
    })
tables.caption(
    pd.DataFrame(rows).style.hide(axis="index"),
    'Принятые в модели Зарафшана параметры водохранилищ.',
    label="reservoir-storage",
)

<a id="tbl-reservoir-config"> </a>

*Таблица 2: Конфигурации резервуаров, показывающие общую емкость, неактивное хранилище, активное хранилище и начальные уровни хранилища.*

<a id="sec-demand-nodes"> </a>

## Узлы сельскохозяйственного водопотребления
ZRB обслуживает шесть основных ирригационных округов, как показано на рисунке [Рисунок 4](#fig-irrigation-districts), каждый из которых смоделирован как узел водопотребления TaqSim. Потребность в воде в этих узлах была получена на основе региональной оценки потребности в оросительной воде с использованием дистанционного зондирования ([Рагеттли и др., 2025 г.](https://doi.org/10.1016/j.ejrh.2025.102185)). Узел спроса абстрагирует весь ирригационный район, как определено в [Рисунок 4](#fig-irrigation-districts), с водоотводами первичных каналов, внутренними сетями вторичных и третичных каналов, инфраструктурой на уровне поля и потреблением воды сельскохозяйственными культурами (эвапотранспирацией). Узел спроса каждого округа вычисляет необходимое отвод как `requirement / efficiency` , где совокупная эффективность является произведением эффективности транспортировки (доля, выживающая во внутренних каналах округа) и эффективности поля (доля, достигающая корней сельскохозяйственных культур). Узел забирает только то, что ему нужно; избыток воды проходит к узлам, расположенным ниже по течению.

Важно отметить, что узлы водопотребления в TaqSim являются **сквозными**: вода, подаваемая сверх потребности района, продолжает двигаться вниз по течению к последующим узлам. Это создает перекрестные соединения в сети – например, избыток воды в Даргоме течет далее в HW_Дамхоза, а избыток в Мирзапае достигает RES_Акдарья – а это означает, что избыточная подача в один район не полностью теряется, но может принести пользу пользователям ниже по течению.  [Раздел 2.2](#sec-taqsim) более подробно объясняет концепцию сквозного узла.

Такое поведение имеет важное аналитическое последствие: вода, поставляемая *сверх* потребности района в любой месяц, не приносит никакой дополнительной выгоды в этом районе — она просто проходит мимо. Мы называем это **чрезмерной поставкой**, а дополнительная концепция **эффективной доставки** учитывает только воду, поступающую в соответствии с потребностями каждого района за определенный период времени. Эти показатели становятся центральными в сравнении стратегий в [Раздел 5.4](#sec-comparison), где они показывают, что наивное распределение может привести к затоплению некоторых районов и одновременному лишению воды в других.

[Таблица 3](#tbl-demand-config) перечисляет шесть ирригационных районов и их параметры эффективности. Эти значения взяты из литературы ФАО по эффективности орошения ([ФАО, 1989 г.](https://www.fao.org/3/T7202E/T7202E00.htm)) и адаптированы для условий Узбекистана. Излишне говорить, что эти общие значения можно будет лучше адаптировать к местным условиям и характеристикам отдельных районов, если станет доступной более качественная информация.

In [ ]:
# @title Узлы сельскохозяйственного водопотребления · код
demand_configs = load_demand_config()
dem_rows = []
for dem in demand_configs:
    combined = dem.conveyance_efficiency * dem.field_efficiency
    dem_rows.append({
        'Район': dem.name,
        'КПД транспортировки': f"{dem.conveyance_efficiency:.0%}",
        'КПД полива': f"{dem.field_efficiency:.0%}",
        'Общий КПД': f"{combined:.1%}",
    })
tables.caption(
    pd.DataFrame(dem_rows).style.hide(axis="index"),
    'Принятые значения эффективности водопользования по ирригационным районам.',
    label="district-efficiencies",
)

<a id="tbl-demand-config"> </a>

*Таблица 3: Конфигурации ирригационных районов, показывающие эффективность транспортировки, эффективность поля и их совокупный продукт.*

<a id="sec-powerplant"> </a>

## Навоийская ТЭЦ
Навоийская теплоэлектростанция (см. [Рисунок 3](#fig-navoi-intake)) является критически важным промышленным объектом в нижнем течении ЗРБ ([Монитор, 2024 год](https://www.gem.wiki/Navoi_power_station)). В отличие от сельскохозяйственных нужд, которые потребляют воду за счет испарения, электростанции требуется непотребляемая охлаждающая вода — вода проходит через систему охлаждения и возвращается в реку. Для безопасной работы требуется минимальный поток 15 m<sup>3</sup>/s (приблизительно 1,30 Mm<sup>3</sup>/day по суточным часам модели) (Джованоли, 2025). Без достаточного количества охлаждающей воды турбины могут перегреться, выработку электроэнергии придется сократить, а оборудование может быть повреждено. Это делает электростанцию ​​серьезным препятствием для доставки воды: в отличие от сельскохозяйственного дефицита, который постепенно снижает урожайность, нехватка охлаждающей воды вызывает немедленные эксплуатационные проблемы.

В TaqSim электростанция моделируется как узел PassThrough с пропускной способностью 30 m<sup>3</sup>/s и минимальным потоком 15 m<sup>3</sup>/s. Системные информационные панели в [Раздел 5](#sec-baseline-simulations) отслеживают расход электростанции в соответствии с этим пороговым значением на каждом этапе моделирования, показывая, как различные стратегии распределения влияют на надежность охлаждающей воды. В сопутствующем учебнике по оптимизации специальная целевая функция (см. [Раздел 12.1](#sec-taqsim-objectives)) позволяет провести формальный анализ компромисса между надежностью электростанции и другими конкурирующими целями.

<a id="sec-sinks"> </a>

## Конечные выходные узлы и межбассейновые переброски
Три конечных узла Sink в модели ZRB — Навои, Джизак и Кашкадарья — представляют собой воду, покидающую смоделированную область ([Рагеттли и др., 2025 г.](https://doi.org/10.1016/j.ejrh.2025.102185)). Два межбассейновых переноса — в Джизак и Кашкадарью — видны как заштрихованные серым регионы в [Рисунок 2](#fig-canal-network). В отличие от потерь (которые приводят к непродуктивному удалению воды), отток через выходные узлы представляют собой **преднамеренные обязательства**: они обеспечивают снабжение населения, сельского хозяйства и экосистем за пределами изучаемой территории.

| Выходной узел | Пункт назначения | Цель | Ключевое ограничение |
|:---------------|:------------------|:---------------|:----------------------|
| **Выходной узел_Навои** | Бухарский оазис (нижнее течение) | Орошаемое земледелие, городское снабжение ( [Каримов и др., 1995 г.](https://www.fao.org/4/v9529e/v9529e06.htm) ) | Также обслуживает Навоийскую электростанцию ​​([Раздел 3.5](#sec-powerplant)); вдвойне чувствителен к распределению |
| **Выходной узел_Джизак** | Джизакская область (бассейн Сырдарьи) | Межбассейновый переброс для орошения ( [САВатер-Инфо, 2024 г.](https://www.cawater-info.net/zeravshan/water_e.htm) ) | Регулируется межпровинциальными соглашениями; не подлежит обсуждению краткосрочные |
| **Выходной узел_Кашкадарья** | Кашкадарьинская область (юг) | Межбассейновый переброс для орошения ( [САВатер-Инфо, 2024 г.](https://www.cawater-info.net/zeravshan/water_e.htm) ) | Местная река недостаточна; трансферт является структурной необходимостью |

На эти три стока в совокупности приходится значительная доля общей пропускной способности системы, часто сравнимая по величине с потреблением района (см. таблицы водного баланса в [Раздел 5](#sec-baseline-simulations)). Любая стратегия распределения, которая уменьшает отток через выходные узлы, косвенно перераспределяет воду из расположенных ниже по течению и соседних провинций. С точки зрения моделирования, выходные узлы служат **граничными условиями**, при этом минимальные требования к расходу отражают нижнюю границу приемлемого оттока. Различия в оттоке через выходные узлы между стратегиями показывают, как каждая из них уравновешивает спрос в верхнем течении и межбассейновые обязательства – компромисс, который в конечном итоге требует политических переговоров, а не только гидравлической оптимизации.

<details style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #3fb618; background-color: #edf7ea;"> 
 <summary> <strong> &#x1F4A1; Совет: Упражнение 1. Исследование сети </strong> </summary>

Используя функцию `load_edges()`, найдите **все различные пути** от `Source` до `Sink_Navoi`.

**Задача:** Написать код для загрузки конфигурации периферии и перечисления всех маршрутов от `Source` до `Sink_Navoi` . Сколько существует различных маршрутов? Какой путь проходит через наибольшее количество узлов?

**Подсказка:** Вам потребуется импортировать функцию: `from zarafshan_taqsim import load_edges` . Данные о рёбрах сети — это словарь, сопоставляющий каждый узел со списком нижестоящих целей. Здесь хорошо работает рекурсивный поиск в глубину. Следите за узлами, которые появляются как цели для нескольких вышестоящих узлов.

**Ожидаемый результат**: вы должны найти несколько различных путей (порядка 5–10). Самый длинный путь проходит через 7--8 узлов. Пути расходятся у первого разветвителя HydroWorks и вновь сходятся через сквозные соединения узла водопотребления.

</details> 
 <div style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #d63384; background-color: #fce4f0;"> 
 <strong> &#x2757; Важно: основные выводы — модель системы </strong> <br> <br>

- Модель ZRB объединяет источники, разделители, резервуары, узлы потребления, проходную станцию ​​ниже по течению, конечные выходные узлы и промежуточные транспортные сегменты, которые транспортируют воду и несут потери.
- При эксплуатации водохранилища используется четырехзонный SLOP: мёртвый объём (без сброса), буфер (сниженный), консервация (целевой уровень), контроль наводнений (избыточный сброс).
- Узлы спроса являются **сквозными**: избыточная вода продолжает течь вниз по течению, создавая взаимозависимость в сети.
- Три стока представляют собой обязательства по межбассейновой передаче воды в Бухару, Джизак и Кашкадарью – не отходы, а обязательства.

</div> 
Определив топологию системы, мы теперь обратимся к историческим данным, которые ее определяют: сколько воды поступает в систему, когда спрос достигает пика и как выглядит инфраструктура канала.

<a id="sec-historical-data"> </a>

# Исторические данные
 <a id="sec-data-sources"> </a>

## Источники данных и сезонные закономерности
Ежедневное моделирование охватывает период 2017–2022 годов. Приток реки рассчитывается на основе данных ежедневных наблюдений на гидрометрической станции Раватхоза, а потребность в орошении, осадки, испарение и минимальные требования к экологическому стоку извлекаются из наборов исторических данных, собранных в [Рагеттли и др. (2025)](https://doi.org/10.1016/j.ejrh.2025.102185), и при необходимости переводятся в ежедневные входные данные модели. На рисунке ниже представлен **обзор исходных данных за месяц**, используемый для обобщения сезонной структуры поставок и обязательств перед переходом к ежедневным результатам моделирования.

Гидрология Центральной Азии демонстрирует сильные сезонные закономерности ([Гролл и др., 2015 г.](https://doi.org/10.1007/s12665-013-2988-5)). Весна (март-май) приносит с собой таяние снегов и повышение притоков. Летом (июнь-август) наблюдается максимальный приток талых снеговых вод, а позднее в этом сезоне таяние ледников, совпадающее с пиком потребности в орошении. Осенью (сентябрь-ноябрь) наблюдается снижение стока и сезон сбора урожая, а зимой (декабрь-февраль) наблюдается низкий сток и минимальный спрос.

In [ ]:
# @title Источники данных и сезонный ход · код
inflows = load_inflow()

# Распределённый сток от шести источников, питаемых осадками
precip = load_precipitation()
rf_configs = load_runoff_config()
total_runoff = [
    sum(p * rf.area * rf.runoff_coefficient * 1000 for rf in rf_configs)
    for p in precip
]

# Обязательства по межбассейновым переброскам в конечных узлах
transfers = {
    "Navoi": load_min_flow("navoi"),
    "Jizzakh": load_min_flow("jizzakh"),
    "Kashkadarya": load_min_flow("kashkadarya"),
}

fig, axes = plot_supply_demand(
    inflow=inflows,
    districts=DEFAULT_DISTRICTS,
    transfers=transfers,
    runoff=total_runoff,
    start_year=2017,
    figsize=(10, 7),
    show_balance=True,
)
figures.caption(
    fig,
    'Обзор месячных исходных данных о притоке и обязательствах. Показаны общий приток, потребности орошения и межбассейновых перебросок с накоплением и итоговый баланс притока и обязательств.',
    label="supply-demand",
)

[Рисунок 6](#fig-supply-demand) обобщает ежемесячные исходные данные, лежащие в основе ежедневной модели. Общий приток системы в среднем составляет примерно 4,7 Bm<sup>3</sup>/yr, из которых 3,96 Bm<sup>3</sup>/yr поступает через Зеравшан в Раватхозу, а 0,71 Bm<sup>3</sup>/yr поступает в виде распределенного стока осадков на шесть притоковых водосборных бассейнов. Спрос на ирригацию во всех шести округах составляет примерно 1,14 Bm<sup>3</sup>/yr – всего около 24% притока. Однако система должна также выполнять значительные обязательства по межбассейновым переброскам: 0,78 Bm<sup>3</sup>/yr в Навои (Бухарский оазис), 0,34 Bm<sup>3</sup>/yr в Кашкадарью и 0,19 Bm<sup>3</sup>/yr в Джизак (бассейн Сырдарьи). В совокупности **общие обязательства достигают примерно 2,5 Bm<sup>3</sup>/yr , или 53% притока**. Оставшиеся 47% должны покрывать потери при транспортировке воды воды по каналу (фильтрация, испарение, эксплуатационные разливы), испарение из водохранилища и изменения в хранении, что оставляет мало места для ошибки в том, как распределяется вода после моделирования системы на ежедневном временном интервале.

На рисунке показано, что дефицит воды может возникнуть весной и в конце лета на уровне системы. Это требует тщательного управления водами реки, включая сроки доставки, а также работу двух водохранилищ.

<details style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #3fb618; background-color: #edf7ea;"> 
 <summary> <strong> &#x1F4A1; Совет: Упражнение Б. Арифметика спроса и предложения </strong> </summary>

Используя цифры из [Рисунок 6](#fig-supply-demand), ответьте на следующее:

1. Если потери при транспортировке воды по каналу составляют примерно 35% от общего притока, сколько воды (в Bm<sup>3</sup>/yr ) теряется в каналах? Это больше или меньше общей потребности в орошении?
2. Суммарная емкость водохранилищ (Каттакурган + Акдарья) составляет примерно 773 Mm<sup>3</sup>. Выразите это как часть общего годового притока. Сколько месяцев среднего притока могут храниться в водохранилищах?
3. Если бы вы могли устранить все потери в каналах, был бы общий приток достаточным для покрытия всех обязательств (орошение + межбассейновые переброски) в каждом месяце ряда исходных данных? Почему или почему нет?

**Подсказка:** Общий приток равен \ <sub> 4.7 Bm<sup>3</sup>/yr . По вопросу 3 учтите, что валовые потребности в водозаборе (потребность/эффективность) примерно в два раза превышают потребность сельскохозяйственных культур в воде, и что несколько районов и водосборников одновременно конкурируют за один и тот же сток.

**Ожидаемый результат:** (1) Потери в канале \ </sub> 1,6 Bm<sup>3</sup>/yr , что превышает общую потребность в орошении (\ <sub> 1,14 Bm<sup>3</sup>/yr ). (2) Емкость водохранилища составляет \ </sub> 16% годового притока, или примерно 2 месяца среднего притока. (3) Нет — даже без потерь общие обязательства составляют ~53% от притока, а валовые потребности в заборе у ворот округа примерно вдвое превышают потребность в урожае из-за неэффективности транспортировки; в пиковые месяцы конкурирующие водозаборы в сети могут превышать доступное предложение.

</details> 
 <a id="sec-canal-properties"> </a>

## Свойства канала и конфигурация потерь
Сеть каналов ЗРБ представлена множеством транспортных участков, соединяющих функциональные узлы. Каждый сегмент имеет физические свойства, определяющие потери при транспортировке воды: длина (км), средняя ширина водной поверхности (м), тип обделки (земляная, частичная или бетонная) и состояние инфраструктуры (хорошее, среднее или плохое). Топология сети и пропускная способность каналов взяты из схемы магистрального канала, показанной в [Рисунок 2](#fig-canal-network), предоставленной Министерством водного хозяйства Узбекистана [MoWR; Джованоли (2025).

Проектная пропускная способность каналов в сети взяты из [Рисунок 2](#fig-canal-network). Это максимальный поток, на который рассчитан каждый сегмент магистрального канала. Это ограничения инфраструктуры, а не наблюдаемые потоки. Имитационная модель использует эту проектную пропускную способность в качестве ограничений пропускной способности рёбер сети, гарантируя, что моделируемый поток не может превысить то, что может передать физическая инфраструктура.

Однако **физические свойства каждого канала, используемые для моделирования потерь, являются оценочными значениями**: длины были измерены с помощью ГИС, ширина была масштабирована с учетом расчетной пропускной способности, типы обделок были назначены на основе местных знаний и исследований инфраструктуры, в результате чего получилось примерно 63% земляных каналов, что несколько ниже среднего показателя по стране в 75% ([Ходжиев и Атаджанова, 2022 г.](https://doi.org/10.1051/e3sconf/202336501011)), что отражает то, что основные магистральные каналы в Зарафшанской системе частично облицованы, а условия по умолчанию были установлены на «средний» при отсутствии данных полевых исследований. Общая эффективность сети составляет примерно 63% ([Ходжиев и Атаджанова, 2022 г.](https://doi.org/10.1051/e3sconf/202336501011)). Потери в канале анализируются в [Раздел 5.4.2](#sec-canal-losses).

In [ ]:
# @title Свойства каналов и параметры потерь · код
canal_props = load_canal_properties()
canal_df = pd.DataFrame.from_dict(canal_props, orient='index')
canal_df.index.name = 'edge_id'

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

lining_counts = canal_df['lining'].value_counts()
colors = {'earthen': '#e67e22', 'partial': '#3498db', 'concrete': '#2ecc71'}
axes[0].pie(lining_counts.values, labels=lining_counts.index, autopct='%1.0f%%',
            colors=[colors[l] for l in lining_counts.index], startangle=90)
axes[0].set_title('Распределение типов облицовки')

for lining in ['earthen', 'partial', 'concrete']:
    subset = canal_df[canal_df['lining'] == lining]
    axes[1].scatter(subset['length_km'], subset['avg_width_m'],
                    c=colors[lining], label=lining, alpha=0.7, s=60)
axes[1].set_xlabel('Длина канала (км)')
axes[1].set_ylabel('Ширина канала (м)')
axes[1].set_title('Размеры по типу облицовки')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
figures.caption(
    fig,
    'Свойства каналов: распределение типов облицовки и размеры каналов по типу облицовки. Преобладание каналов в земляном русле объясняет высокие потери при транспортировке воды.',
    label="canal-properties",
)

Как показывает [Рисунок 7](#fig-canal-properties), большинство каналов в сети земляные (не облицованные), что является основной причиной потерь на утечку. Диаграмма рассеяния показывает, что земляные каналы, как правило, длиннее и шире, чем каналы с облицовкой, что усугубляет проблему потерь.

<div style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #d63384; background-color: #fce4f0;"> 
 <strong> &#x2757; Важно: основные выводы: исторические данные </strong> <br> <br>

- Общий средний приток \ <sub> 4,7 Bm<sup>3</sup>/yr ; общие обязательства (орошение + межбассейновые переводы) составляют \ </sub> 2,5 Bm<sup>3</sup>/yr (53% притока).
- Сильная сезонность: ледники и таяние снегов приводят к летнему пику поставок, который в целом совпадает с пиковым спросом на ирригацию, но система перегружена, поэтому водохранилищам приходится буферизовать месяцы, когда конкурирующие заборы и потери превышают доступный сток.
- Сеть каналов преимущественно земляная (~63%), свойства каналов оцениваются на основе ГИС и литературы, а не полевых исследований.
- Система действительно ограничена в воде: после потерь существует ограниченная вероятность ошибки распределения.

</div> 
Анализ спроса и предложения подтверждает, что ЗРБ ограничен в воде: обязательства превышают 50% притока, а потери в канале поглощают еще треть. Вопрос больше не в том, имеет ли значение распределение, а в том, насколько оно велико. Теперь мы тестируем три все более сложные стратегии разделения в этой системе.

<a id="sec-baseline-simulations"> </a>

# Базовые расчёты
Определив топологию системы, входные данные и инфраструктуру канала, мы теперь исследуем, как вода течет через сеть при различных правилах эксплуатации. Мы приступаем к трем все более сложным стратегиям разделения для распределения доступных потоков по системе — единообразным, статически настроенным и меняющимся помесячно — прежде чем сравнивать их друг с другом. Все модели включают реалистичные потери в каналах по всей транспортной сети. Это помогает нам получить хорошее представление о системе водоснабжения и о том, как различные правила распределения определяют результаты с точки зрения удовлетворения спроса.

Затем в сопутствующем учебнике по оптимизации мы берем эту водохозяйственную систему и изучаем различные модели распределения воды, которые оптимизированы с учетом набора поставленных целей.

<a id="sec-uniform-baseline"> </a>

## Базовый сценарий равного распределения
В едином базовом сценарии используется одинаковое разделение на каждом узле распределения HydroWorks. Например, HW_Ravatkhoza поровну распределяет приток между четырьмя своими ветвями, расположенными ниже по течению, несмотря на то, что некоторые ветви обслуживают крупные ирригационные районы, в то время как другие питают водоприемники с низким спросом (см. сеть в [Рисунок 2](#fig-canal-network)). Это простейшее возможное правило работы, которое служит отправной точкой. Это также довольно глупое действующее правило, как мы увидим в свое время. Что касается степеней свободы, единообразное правило имеет **нулевые свободные параметры**: каждое соотношение фиксируется на значении $1/n$ (где $n$ — количество исходящих ветвей), поэтому настраивать нечего. Это делает эту гипотезу полезной, но это также означает, что правило совершенно нечувствительно к структуре спроса на нисходящем рынке.

Следующий блок кода описывает полный рабочий процесс моделирования и анализа. Последующие симуляции следуют той же схеме и показаны со свернутым кодом.

In [ ]:
# @title Шаг 1: создаём систему с учётом потерь в каналах
# Шаг 1: создаём систему с учётом потерь в каналах
system = create_zrb_system(use_canal_losses=True)

# Шаг 2: выполняем суточный расчёт за весь период 2017–2022 годов
system.simulate(TIMESTEPS)

# Шаг 3: задаём узлы для анализа
demand_ids = list(DEFAULT_DISTRICTS)       # 6 ирригационных районов
reservoir_ids = ["RES_Kattakurgan", "RES_Akdarya"]
POWERPLANT_MIN_FLOW = 15.0                 # Минимальный порог, м³/с

# Шаг 4: восстанавливаем водный баланс по журналу событий TaqSim
wb = compute_water_balance(system)

<a id="sec-reading-wb"> </a>

### Итоговый водный баланс
Таблица водного баланса представляет собой баланс массы: она отслеживает каждую каплю воды, попадающую в систему, и определяет, где она оказывается. Думайте об этом как о финансовой книге: притоки — это доходы, полезное использование, а потери — это расходы, а невязка баланса показывает, сбалансированы ли бухгалтерские книги.

[Таблица 4](#tbl-water-balance-uniform) показывает водный баланс для однородного базового уровня:

In [ ]:
# @title Полученный водный баланс · код
print_annual_water_balance(wb, timesteps=TIMESTEPS)

<a id="tbl-water-balance-uniform"> </a>

*Таблица 4: Среднегодовой водный баланс для единого базового моделирования, показывающий, куда попадает каждая капля воды в течение всего ежедневного периода моделирования 2017–2022 годов.*

Индивидуальные условия следующие:

**Река + Сток** – это вся вода, поступающая в систему: приток реки Зарафшан на верхней границе плюс боковой сток от осадков в шести узлах притока. Эта сумма является знаменателем для всех значений «% притока» в таблице.

В разделе **Полезное использование** «Потребление районами» — это вода, фактически удаленная из гидрологической системы шестью узлами сельскохозяйственного потребления — вода, которая орошает посевы и испаряется. «Отток через выходные узлы» — это вода, которая достигает трех конечных узлов «Sink», включая два межбассейновых водоперевода и сливной сток в Навои на нижней границе, что обеспечивает поступление воды в Бухарскую область. Слив в раковину не является потерей; он представляет собой экологические потоки, обязательства ниже по течению или воду, покидающую моделируемую область.

В разделе **Потери** учитывается вода, удаленная из системы непроизводительно. Фильтрация из канала — потеря воды через русла каналов и берега в грунтовые воды — рассчитывается с помощью модели $\sqrt{Q}$ и является доминирующим механизмом потерь, как и ожидалось для сети, состоящей преимущественно из земляных каналов ([Заман и др., 2023 г.](https://doi.org/10.1016/j.agwat.2023.108438)). См. [Раздел 5.4.2](#sec-canal-losses). Испарение из водохранилища рассчитывается ежемесячно на основе площади поверхности (полученной на основе кривых высота-объем) и скорости испарения в открытой воде (Shibuo & Jarsjo, 2007); обратите внимание, что скорость испарения из водохранилища выше, чем скорость испарения из канала, поскольку открытые водоемы имеют большую нагрузку и воздействие, чем узкие поверхности каналов. Неэффективность спроса представляет собой потери на уровне поля в узлах спроса: каждый узел водопотребления TaqSim удаляется `requirement / efficiency`, но потребляет только `requirement`; разница записывается как потери из-за неэффективности спроса. Испарение канала и эксплуатационные потери являются второстепенными компонентами, а краевой перелив улавливает потери воды, когда поток превышает пропускную способность канала.

**Резервуар**$\Delta$ Хранение — это чистое изменение объема воды, хранящейся за период моделирования ( $S_{\text{final}} - S_{\text{initial}}$ ). При многолетнем моделировании это должно быть небольшим.**Невязка баланса** ( $\text{Inflow} - \text{Consumed} - \text{Sink outflow} - \text{Losses} - \Delta\text{Storage}$ ) подтверждает сохранение массы: значение, близкое к нулю, указывает на то, что моделирование правильно учитывает всю воду.

[Таблица 5](#tbl-district-uniform) показывает удовлетворение спроса при едином базовом сценарии эксплуатации.

In [ ]:
# @title Полученный водный баланс · код
_ = print_district_summary(system, demand_ids, timesteps=TIMESTEPS)
print()
print_powerplant_stats(system, timesteps=TIMESTEPS)

<a id="tbl-district-uniform"> </a>

*Таблица 5: Удовлетворение спроса по районам при едином базовом сценарии с указанием годового спроса, поставок, дефицита и процента удовлетворения.*

Чтение этой таблицы требует внимательности, поскольку на первый взгляд кажется, что некоторые районы получают гораздо больше воды, чем им требуется, но при этом все еще наблюдается значительный дефицит. Три фактора объясняют этот очевидный парадокс:

1. **Потери эффективности поглощают большую часть подаваемой воды.** В столбце «Потребность» указана *потребность сельскохозяйственных культур в воде* — то, что на самом деле необходимо растениям посредством эвапотранспирации. Но при совокупной эффективности транспортировки и поля всего 48,8% ( [Таблица 3](#tbl-demand-config) ) узел должен *выводить* из канала примерно вдвое больше требуемого урожая, чтобы удовлетворить эту потребность. Например, для урожая Даргома требуется 251,5 Mm<sup>3</sup>/yr, но узел должен вывести 251,5/0,488 ≈ 515 Mm<sup>3</sup>/yr, чтобы доставить это количество в корневую зону. В столбце «Доставлено» показано общее количество воды, поступающей *в* узел — 493,3 Mm<sup>3</sup>/yr — что не соответствует потребности в отводе 515 Mm<sup>3</sup>/yr даже на ежегодной основе.

2. **Правило распределения игнорирует сезонные модели спроса.** Хотя ниво-ледниковый режим Зерафшана обеспечивает пиковое снабжение летом, что в целом совпадает с пиковым спросом на ирригацию, единообразное правило разделения направляет фиксированную долю речного стока в каждый рукав независимо от сезона. Зимой, когда спрос низкий, излишки воды проходят к узлам, расположенным ниже по течению, неиспользованными. В пиковые летние месяцы фиксированная доля распределения может оказаться недостаточной для покрытия валовой потребности в заборе воды (потребность/эффективность), поскольку потери в каналах также достигают максимального значения (масштабирование $\sqrt{Q}$). Поскольку узлы водопотребления не имеют хранилища, месяцы с низким спросом не могут компенсировать месяцы с высоким спросом.

3. **Пространственное неравенство при равномерном распределении.** Равное разделение на всех гидроузлах HydroWorks игнорирует совершенно разные величины спроса в разных ветвях сети. Это создает крайнее неравенство: Mirzapay получает 994 Mm<sup>3</sup>/yr притока (для полного вывода требуется только <sub> 367) и достигает 99% удовлетворения, в то время как Narpay получает всего 128 Mm<sup>3</sup>/yr (нужно </sub> 310) и обеспечивает только 15% удовлетворения. Правило равного распределения приводит к затоплению одних ветвей и голоду других.

Статистика расхода воды электростанции, напечатанная под [Таблица 5](#tbl-district-uniform), показывает не менее тревожную ситуацию с использованием воды в несельскохозяйственных целях. Для безопасной работы Навоийской ТЭС требуется минимальный расход охлаждающей воды 15 m<sup>3</sup>/s ([Раздел 3.5](#sec-powerplant)). При равномерном разделении средний поток, поступающий на завод, составляет 12,6 m<sup>3</sup>/s — все еще ниже порогового значения — и завод испытывает недостаток поставок в **1465 из 2191 ежедневных временных интервалов**. Проблема носит структурный характер: Карманская гидростанция ([Рисунок 2](#fig-canal-network), узел 7), питающая ветвь электростанции, находится в нижнем конце сети. При равномерном разделении каждый водозабор вверх по течению уже откачал равную долю независимо от потребностей ниже по течению, в результате чего филиал Кармана хронически недопоставляется. Это прямое следствие правила «равных долей повсюду», применяемого к сети с крайне неравными нисходящими обязательствами.

In [ ]:
# @title Полученный водный баланс · код
fig = plot_system_dashboard(
    wb, system, demand_ids,
    timesteps=TIMESTEPS,
    title="",
    #title="Равные доли — показатели системы (2017–2022)",
)
figures.caption(
    fig,
    'Показатели базового сценария равного распределения за суточный расчёт 2017–2022 годов: суточные расходы и потери, накопленный водный баланс, потребность и подача по районам, расход у ТЭС и его минимальный порог.',
    label="dashboard-uniform",
)

[Рисунок 8](#fig-dashboard-uniform) обеспечивает четырехпанельную диагностику базового сценария равного распределения. Важное уточнение: «Поставка» в Панелях 1 и 2 — это общий объем воды *полученный только в шести узлах сельскохозяйственного потребления* — он **не** включает стоки в три конечных стока (Навои, Джизак, Кашкадарья). Отток через выходные узлы представляет собой отдельное существенное обязательство (<sub> 29% притока), которое конкурирует с сельскохозяйственными поставками за такое же ограниченное предложение.

**Панель 1 (Ежедневный приток/подача/потери)** показывает ежедневный временной ряд за весь период моделирования 2017–2022 годов. Синяя линия притока демонстрирует характерный ниво-ледниковый сезонный цикл: резкие летние пики, вызванные таянием снегов и ледниковым стоком, а также низкий зимний базовый сток. Зеленая линия подачи отслеживает, сколько воды фактически поступает в шесть ирригационных районов каждый день. При равномерном распределении поставки следуют той же сезонной форме, что и приток, но на гораздо более низком уровне. Разрыв между притоком и доставкой объясняется тремя стоками: потерями при передаче по каналу (красная линия), оттоком через выходные узлы (отдельно не показаны) и изменениями объема водохранилища. Красная линия потерь показывает, что потери сами по себе являются сезонными, достигая пика летом, когда больше воды протекает через сеть земляных каналов и, следовательно, больше воды теряется на фильтрацию и испарение. Даже на суточном временном интервале сохраняется та же основная закономерность: во время пикового ирригационного сезона поставляемая доля по-прежнему значительно отстает от того, что в совокупности необходимо шести районам.

**Панель 2 (Совокупный водный баланс)** показывает те же три количества, накопленные с течением времени. Растущий вертикальный разрыв между совокупным притоком (синий) и совокупным расходом (зеленый) представляет собой общий объем воды, поступившей в систему, но так и не достигшей узла потребления — вместо этого она была потеряна при транспортировке, доставлена ​​в выходные узлы или поглощена резервуарами-хранилищами. К концу моделирования совокупные потери составляют примерно одну треть совокупного притока, в то время как совокупная доставка к узлам водопотребления составляет лишь около одной четверти. Остальная часть выходит через три выходные узлы. Устойчивый рост всех трех кумулятивных кривых подтверждает, что система работает в квазистационарном сезонном цикле: водохранилища наполняются и осушаются в течение многих лет, но систематического набора или потери воды в период 2017-2022 годов не происходит.

Хотя спрос на сельскохозяйственную продукцию составляет лишь</sub>24% общего притока, система должна также обслуживать существенные межбассейновые обязательства по перекачке в Навои (Бухарский оазис), Джизак и Кашкадарью, что повышает общие обязательства по<sub>53% притока (см.[Рисунок 6](#fig-supply-demand)). После того, как потери при транспортировке воды по каналу поглощают еще</sub>35%, система действительно перегружена обязательствами. При равномерном разделении ситуация усугубляется серьезным пространственным неравенством: некоторые районы получают гораздо больше воды, чем им нужно, в то время как другие сталкиваются с хроническим дефицитом, поскольку равное распределение игнорирует совершенно разные величины спроса в разных ветвях сети.

Возникает естественный вопрос: можем ли мы добиться большего, просто изменив правила?

<details style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #3fb618; background-color: #edf7ea;"> 
 <summary> <strong> &#x1F4A1; Совет: Упражнение C. Прогнозирование перед моделированием </strong> </summary>

**Делление, пропорциональное потребности** означает, что вместо равномерного разделения стока между всеми расположенными ниже по течению ветвями на каждом гидроузле каждая ветвь получает долю, пропорциональную общей потребности в воде после нее. Филиал, обслуживающий большой ирригационный район, получает большую долю, чем филиал, обслуживающий небольшой сток.

Прежде чем читать следующий раздел, сделайте прогнозы на основе того, что вы уже узнали:

1. Если мы перейдем от равного распределения к разделению, пропорциональному спросу, будут ли **общие потери в каналах** увеличиваться, уменьшаться или оставаться примерно такими же? Объясните свои рассуждения.
2. Будет ли **Навоийская электростанция** получать больше или меньше охлаждающей воды при пропорциональном распределении потребности? (Подсказка: подумайте, что означает пропорциональность спроса для сплиттера HW_Karmana, который обслуживает как район Карманаконимекс, так и филиал электростанции.)
3. Может ли какая-либо стратегия разделения полностью устранить сельскохозяйственный дефицит, учитывая, что общие обязательства составляют \ <sub> 53% притока, а потери в каналах поглощают \ </sub> 35%?

**Прочитав** разделы 4.2–4.4, вернитесь к своим прогнозам. Какие были правильными? Что вас удивило?

**Ожидаемый результат:** (1) Общие потери в каналах остаются примерно такими же: потери являются физическим свойством инфраструктуры канала, а не правилом распределения. (2) Электростанция получает *больше* воды, поскольку HW_Karmana теперь распределяет ее пропорционально потребностям отрасли электростанции. (3) Никакая стратегия разделения не может устранить дефицит: общие обязательства (\ <sub> 53%) плюс убытки (\ </sub> 35%) составляют почти 90% притока, оставляя недостаточную маржу для одновременного полного удовлетворения всех округов.

</details> 
 <a id="sec-static-baseline"> </a>

## Базовый сценарий с постоянными настроенными долями
Простейшим улучшением по сравнению с равномерным разделением является вычисление одного фиксированного коэффициента разделения на филиал на каждом предприятии HydroWorks на основе общего долгосрочного спроса, суммированного за весь период моделирования 2017–2022 годов:

$$r_i = \frac{\sum_{t=1}^{T} D_i(t)}{\sum_j \sum_{t=1}^{T} D_j(t)}$$

где:

- $r_i$ — это **коэффициент разделения**, назначенный филиалу $i$ в данном HydroWorks — фиксированная доля притока, которую филиал $i$ получает на каждом временном шаге,
- $D_i(t)$ — это **общая потребность в нисходящем направлении**, достижимая через ветвь $i$ на временном шаге $t$, суммированная по всем узлам водопотребления (ирригационные районы, стоки и электростанция), которые лежат ниже по течению от ветви $i$ в топологии сети,
- индекс $t$ охватывает все временные интервалы ежедневного периода моделирования 2017–2022 гг.,
- индекс $j$ охватывает **все ветви**, оставляя один и тот же разделитель HydroWorks, поэтому знаменателем является общая потребность во всех последующих ветвях, обеспечивая сумму коэффициентов, равную 1.

Вычисления работают следующим образом: для каждого разветвителя HydroWorks алгоритм обходит сеть вниз по течению вдоль каждой исходящей ветви и накапливает потребность каждого достижимого узла потребности, приемника и электростанции. Суммирование этих потребностей по всему моделированию дает одно число для каждой отрасли, представляющее ее общую долгосрочную потребность в воде. Коэффициент $r_i$ представляет собой долю этого филиала в общих обязательствах разделителя. Филиал, обслуживающий такой крупный ирригационный район, как Даргом, получит соответственно большую долю, чем филиал, питающий небольшой сток.

Каждый сплиттер использует одно и то же соотношение каждый день — в самой политике нет сезонных изменений. Это эквивалентно вопросу: «За весь исторический период какую долю общего спроса на переработку обслуживает каждая отрасль?» и соответственно распределяем. Функция `apply_static_demand_splitting()` вычисляет эти коэффициенты, используя ту же топологию дерева спроса, что и функция ежемесячного изменения, но сжимает измерение времени путем суммирования. Минимальный уровень в 5% предотвращает полное истощение любой ветви, гарантируя, что даже ветви с низким спросом получат некоторое количество воды.

Стоит сделать паузу, чтобы подсчитать **степени свободы**в задаче распределения, поскольку они определяют размерность пространства параметров, которое оптимизатор может позже изучить. На каждом разветвителе с исходящими ветвями $n$ коэффициенты разделения должны в сумме равняться 1, оставляя $n - 1$ независимые параметры. В базового сценария равного распределения из предыдущего раздела все передаточные числа зафиксированы на уровне $1/n$ , что дает**нулевую степень свободы** — нет ручек, которые можно было бы поворачивать. В статически настроенном подходе шесть активных разветвителей HydroWorks (HW_Confluence — это сквозной канал с одной целью) вносят в общей сложности независимые коэффициенты $3 + 2 + 3 + 1 + 2 + 1 = 12$. Статически настроенное правило выбирает одну конкретную точку в этом 12-мерном пространстве параметров, определяемую данными спроса. В представленном далее подходе с ежемесячным изменением ( [Раздел 5.3](#sec-monthly-baseline) ) каждый из этих 12 коэффициентов может принимать разные значения в каждом из 12 календарных месяцев, расширяя пространство параметров до измерений $12 \times 12 = 144$. Это измерения, которые оптимизатор может искать, чтобы найти модели распределения, которые наилучшим образом сбалансируют конкурирующие цели — к этому моменту мы возвращаемся в [Раздел 12.2](#sec-taqsim-optimisation) и в сопутствующем учебнике по оптимизации.

[Рисунок 9](#fig-static-ratios) визуализирует вычисленные коэффициенты разделения на сетевом графике. Ширина кромок пропорциональна соотношению, назначенному на каждом HydroWorks, а процентные метки показывают долю притока, направляемого в каждое ответвление. Сравните это с единым базовым уровнем ( [Рисунок 5](#fig-network) ), где каждая ветвь, выходящая из сплиттера, получит равную долю (например, по 25% каждая в четырех филиалах Раватхозы). Соотношения пропорциональности спроса поразительно неравны: филиалы, обслуживающие крупные агрегаты спроса вниз по течению, такие как HW_Ravatkhoza → HW_AkKaraDarya, который питает всю сеть среднего бассейна, получают львиную долю, в то время как филиалы с низким спросом, такие как Sink_Jizzakh, получают только минимальный минимальный уровень в 5%.

In [ ]:
# @title Базовый сценарий с постоянными долями · код
fig = plot_static_ratio_network()
figures.caption(
    fig,
    'Постоянные настроенные доли распределения в сети Зарафшана. Толщина рёбер у гидроузлов HydroWorks пропорциональна доле распределения; проценты показывают долю притока в каждую ветвь.',
    label="static-ratios",
)

In [ ]:
# @title Базовый сценарий с постоянными долями · код
system_static = create_zrb_system(use_canal_losses=True)
apply_static_demand_splitting(system_static, verbose=False)
system_static.simulate(TIMESTEPS)
wb_static = compute_water_balance(system_static)

In [ ]:
# @title Базовый сценарий с постоянными долями · код
print_annual_water_balance(
    wb_static,
    timesteps=TIMESTEPS,
    title='Постоянные доли — среднегодовой водный баланс',
)

<a id="tbl-water-balance-static"> </a>

*Таблица 6: Среднегодовой водный баланс для статически настроенного базового уровня.*

[Таблица 6](#tbl-water-balance-static) показывает, что общая структура водного баланса остается в целом схожей с единым базовым уровнем: общий приток, общие потери и отток через выходные узлы практически не изменились, поскольку они определяются физической инфраструктурой и гидрологией, а не правилами распределения. Ключевое различие заключается в *распределении* поставок по узлам водопотребления: тот же общий объем воды достигает узлов водопотребления, но теперь он распределяется более справедливо. Примечательно, что переполнение границ *увеличивается* по сравнению с единой базовой линией (с <sub> 1,45 Bm<sup>3</sup> до </sub> 1,60 Bm<sup>3</sup>). Это противоречит здравому смыслу, но имеет четкое физическое объяснение: разделение, пропорциональное спросу, концентрирует больше воды на ответвлениях, обслуживающих районы с высоким спросом, а некоторые из этих сегментов каналов имеют проектную пропускную способность ([Рисунок 2](#fig-canal-network)), которая слишком мала, чтобы нести увеличенный поток.

При равномерном разделении преобладающий разлив произошел на участке **HW_Раватхоза → Выходной узел_Джизак**( <sub> 1,286 Mm<sup>3</sup> ), поскольку при равном разделении 25% полного речного стока направлялось в канал, рассчитанный всего на 45 m<sup>3</sup>/s. Статическое разделение полностью устраняет эту проблему, предоставляя Джизаку небольшую долю, пропорциональную спросу. Однако в других местах это создает новые узкие места:**HW_Damkhoza → HW_Narpay**(671 Mm<sup>3</sup> , пропускная способность канала </sub> 80 m<sup>3</sup>/s ) теперь переполняется, поскольку Дамхоза направляет больше воды в сторону поддерева Нарпай с высоким спросом, а**HW_Karmana → Powerplant** (926 Mm<sup>3</sup> , впускная способность ~30 m<sup>3</sup>/s ) переполняется, потому что доля, пропорциональная спросу, выделенная ветке электростанции, превышает то, что ее структура всасывания может физически передать в месяцы пикового стока.

In [ ]:
# @title Базовый сценарий с постоянными долями · код
_ = print_district_summary(
    system_static,
    demand_ids,
    timesteps=TIMESTEPS,
    title='Постоянные доли — показатели по районам',
)
print()
print_powerplant_stats(system_static, timesteps=TIMESTEPS, label='Постоянные доли — ТЭС')

<a id="tbl-district-static"> </a>

*Таблица 7: Удовлетворение спроса по районам в соответствии со статически настроенным базовым сценарием.*

[Таблица 7](#tbl-district-static) показывает, что даже это минимальное изменение — 12 фиксированных коэффициентов, полученных из долгосрочного спроса — существенно снижает крайнее пространственное неравенство, наблюдаемое при равномерном разделении. В ранее голодающих районах, таких как Нарпай и Мианкалтосс, наблюдается повышение удовлетворенности, в то время как ранее испытывающие избыток воды районы, такие как Мирзапай, отказываются от лишней воды, которая им не нужна. Однако перераспределение не является идеальным: поскольку соотношения статичны, они не могут учитывать сильную сезонность спроса на ирригацию. Летние месяцы по-прежнему приводят к дефициту даже в районах, которые хорошо снабжаются в годовом исчислении.

Статистика электростанции показывает важный компромисс. Согласно текущим результатам ежедневного моделирования, правило статической настройки **не**повышает надежность электростанции: средний расход падает до 10,1 m<sup>3</sup>/s , а установка находится ниже порога 15 m<sup>3</sup>/s в**1723 из 2191 ежедневных временных интервалов**. Другими словами, настройка коэффициентов разделения для совокупного спроса в нижнем течении улучшает справедливость распределения сельскохозяйственных ресурсов, но все же может ухудшить обслуживание несельскохозяйственного пользователя, расположенного ниже по течению, чья минимальная потребность в стоке не масштабируется с спросом на ирригацию таким же образом.

In [ ]:
# @title Базовый сценарий с постоянными долями · код
fig = plot_system_dashboard(
    wb_static, system_static, demand_ids,
    timesteps=TIMESTEPS,
    title="",
    #title="Постоянные доли — показатели системы (2017–2022)",
)
figures.caption(
    fig,
    'Показатели сценария с постоянными настроенными долями за суточный расчёт 2017–2022 годов. Доли не меняются во времени и пропорциональны суммарной потребности ниже по течению каждой ветви HydroWorks.',
    label="dashboard-static",
)

Однако статические коэффициенты не могут адаптироваться к сезонным тенденциям спроса: одно и то же соотношение применяется в летние месяцы с высоким спросом и в зимние месяцы с низким спросом. Можем ли мы добиться большего, изменяя соотношения с течением времени?

<a id="sec-monthly-baseline"> </a>

## Базовый сценарий с сезонными долями
Вместо одного фиксированного коэффициента мы теперь используем **сезонно меняющийся график распределения**. Потребности в нисходящем направлении сначала суммируются по календарным месяцам, затем расширяются до цикла соотношения дней и лет, который применяется к ежедневным часам моделирования:

$$r_i(m) = \frac{D_i(m)}{\sum_j D_j(m)}$$

где $D_i(m)$ — совокупный спрос на переработку (сельское хозяйство + экология + электростанции), связанный с календарным месяцем $m$ для филиала $i$ . В исполняемой модели эти 12-месячные коэффициенты расширяются до 365-дневного сезонного шаблона и повторяются в ходе ежедневного моделирования. Для ответвлений водохранилищ, в которых отсутствует внутренний спрос, используются прокси-значения (масштабированные доли спроса близлежащего района, например, 50% спроса Мианкалтосса на Каттакурган, 30% спроса Аккарадарьи на Акдарью). Это все еще эвристика, а не оптимизация. Он не учитывает потери каналов, буферизацию водохранилищ или компромиссы между конкурирующими целями. Но это наиболее отзывчивая эвристика перед формальной оптимизацией.

В межсезонье в некоторых филиалах спрос нулевой, что приводит к нулю их коэффициента. Произвольно выбранный минимальный уровень в 5% не позволяет любому филиалу получить нулевое распределение. Это значение является произвольным — выбрано как разумная доля «поддержания активности», достаточно большая, чтобы поддерживать некоторый поток через каждую ветвь, но достаточно маленькая, чтобы не искажать логику пропорциональности спроса в месяцы с высоким спросом. Функция `apply_demand_proportional_splitting()` принимает `min_ratio` в качестве параметра для экспериментов.

In [ ]:
# @title Базовый сценарий с сезонными долями · код
system_monthly = create_zrb_system(use_canal_losses=True)
apply_demand_proportional_splitting(system_monthly, verbose=False)
system_monthly.simulate(TIMESTEPS)
wb_monthly = compute_water_balance(system_monthly)

In [ ]:
# @title Базовый сценарий с сезонными долями · код
print_annual_water_balance(
    wb_monthly,
    timesteps=TIMESTEPS,
    title='Сезонные доли — среднегодовой водный баланс',
)

<a id="tbl-water-balance-monthly"> </a>

*Таблица 8: Среднегодовой водный баланс для ежемесячно меняющегося настроенного базового уровня.*

[Таблица 8](#tbl-water-balance-monthly) показывает, что общие потери остаются сопоставимыми с предыдущими стратегиями — они регулируются физической инфраструктурой, а не правилом распределения. Основное изменение заключается в сокращении избыточной подачи (доли избыточной подачи) и соответствующем увеличении результативности водообеспечения, поскольку график сезонного соотношения направляет воду туда, где она необходима в нужное время года, а не распределяет ее равномерно в течение года.

In [ ]:
# @title Базовый сценарий с сезонными долями · код
_ = print_district_summary(
    system_monthly,
    demand_ids,
    timesteps=TIMESTEPS,
    title='Сезонные доли — показатели по районам',
)
print()
print_powerplant_stats(system_monthly, timesteps=TIMESTEPS, label='Сезонные доли — ТЭС')

<a id="tbl-district-monthly"> </a>

*Таблица 9: Удовлетворение спроса по районам при ежемесячно меняющемся базовом сценарии.*

[Таблица 9](#tbl-district-monthly) демонстрирует дальнейшее улучшение удовлетворенности спроса во всех округах по сравнению со статически настроенным базовым уровнем. Сезонная чувствительность означает, что пиковые периоды орошения – когда правило статики все еще не выполнялось – теперь получают большую долю доступного стока. Однако статистика электростанции движется в противоположном направлении: при сохраненной суточной выработке средний расход падает дальше до 8,7 m<sup>3</sup>/s, а станция находится ниже порога 15 m<sup>3</sup>/s в **1843 из 2191 дневных временных интервалов**.

In [ ]:
# @title Базовый сценарий с сезонными долями · код
fig = plot_system_dashboard(
    wb_monthly, system_monthly, demand_ids,
    timesteps=TIMESTEPS,
    title="",
    #title="Сезонные доли — показатели системы (2017–2022)",
)
figures.caption(
    fig,
    'Показатели сценария с сезонными долями за суточный расчёт 2017–2022 годов. Доли следуют месячному сезонному ходу потребности, применяемому на суточном шаге.',
    label="dashboard-monthly",
)

Коэффициенты, изменяющиеся помесячно, представляют собой наилучшее эвристическое распределение перед формальной оптимизацией для сельскохозяйственных целей. Во время поливного сезона больше воды направляется в сельскохозяйственные отрасли; в периоды более низкого спроса больше стока остается в выходного узла Sinkх и резервуарах. Сезонная чувствительность приводит к существенному сокращению дефицита по сравнению со статической стратегией, и в каждом округе наблюдается повышение удовлетворенности. Ситуация с электростанциями явно неблагоприятна с точки зрения текущей суточной выработки: та же логика сезонного спроса, которая помогает сельскому хозяйству *сжимает* долю электростанций в периоды высокого спроса, поэтому сельскохозяйственные выгоды связаны с худшей надежностью охлаждающей воды в Навои.

<a id="sec-comparison"> </a>

## Сравнение базовых стратегий
Теперь мы сравним все три стратегии, чтобы количественно оценить прогресс от наивного равномерного разделения к распределению с учетом спроса. Напомним, что все три моделирования работают в одной и той же физической системе с идентичными притоком, свойствами канала и характеристиками резервуара — различаются только правила разделения в семи узлах отвода HydroWorks (шесть с несколькими нисходящими ветвями плюс HW_Confluence, одноцелевой проходной узел, не требующий настройки).

[Таблица 10](#tbl-overview) предоставляет номера заголовков. Все три стратегии имеют одинаковый приток и спрос; отличаются только правила разделения. Результативность водообеспечения, которая учитывает только поступление воды туда и тогда, когда она необходима, является самым справедливым показателем. Он повышается от равномерного до изменяющегося помесячно, подтверждая, что месячные соотношения лучше всего соответствуют спросу и предложению. Чрезмерная доставка резко снижается при переходе от единообразных к адаптированным стратегиям, а это означает, что значительная часть речного притока раньше доставлялась не в то место и не в то время.

In [ ]:
# @title Сравнение базовых стратегий · код
tables.caption(
    build_baseline_overview_table(
        wb,
        wb_static,
        wb_monthly,
    ),
    'Сравнение трёх базовых стратегий за весь период: равные, постоянные настроенные и сезонные доли распределения.',
    label="baseline-overview",
)

<a id="tbl-overview"> </a>

*Таблица 10: Количественное сравнение базовых стратегий по ключевым показателям эффективности для полного ежедневного моделирования 2017–2022 годов.*

Следующая визуализация демонстрирует детали этих заголовков.

In [ ]:
# @title Сравнение базовых стратегий · код
fig = plot_baseline_strategy_comparison(
    wb,
    wb_static,
    wb_monthly,
    timesteps=TIMESTEPS,
)
figures.caption(
    fig,
    'Сравнение водного баланса трёх стратегий: равные, постоянные настроенные и сезонные доли; показаны распределение годового притока и основные показатели эффективности.',
    label="comparison-stacked",
)

Верхняя панель [Рисунок 12](#fig-comparison-stacked) показывает, где каждый годовой приток Mm<sup>3</sup> оказывается в рамках трех стратегий. Общий приток одинаков во всех трех системах – любые различия вызваны исключительно тем, как вода распределяется внутри сети. Потребление воды должно увеличиваться по мере улучшения настройки (вода направляется в нуждающиеся в ней районы), а сливной сток и краевой перелив должны уменьшаться. Утечка из канала остается в основном неизменной, поскольку это физическое свойство инфраструктуры канала, а не правило распределения: вы не можете «отключить» фильтрация; для этого требуется прокладка канала ([Ходжиев и Атаджанова, 2022 г.](https://doi.org/10.1051/e3sconf/202336501011)).

На нижней панели показаны три ключевых показателя. **Эффективность системы**(подача к воротам района / общий приток) измеряет валовую пропускную способность, но может вводить в заблуждение, если вода доставляется в районы, которые в ней не нуждаются.**Результативность водообеспечения**(доставка, ограниченная спросом / общий спрос) учитывает доставку только в пределах требований каждого округа за временной интервал. Это самый справедливый показатель, поскольку он не вознаграждает чрезмерную доставку в один район за счет другого.**Доля избыточной подачи** (перерасход/общий приток) – это количество воды, подаваемой сверх потребностей района.

In [ ]:
# @title Сравнение базовых стратегий · код
print("=" * 80)
print('СРАВНЕНИЕ РАЙОНОВ ДЛЯ ТРЁХ СТРАТЕГИЙ')
print("=" * 80)
print()
print_district_comparison(
    system,
    system_static,
    demand_ids,
    timesteps=TIMESTEPS,
    labels=("Uniform", "Static-tuned"),
)
print()
print_district_comparison(
    system_static,
    system_monthly,
    demand_ids,
    timesteps=TIMESTEPS,
    labels=("Static-tuned", "Monthly-tuned"),
)

<a id="tbl-comparison-districts"> </a>

*Таблица 11: Сравнение удовлетворенности спроса по районам по трем стратегиям.*

In [ ]:
# @title Сравнение базовых стратегий · код
print("=" * 80)
print('СРАВНЕНИЕ ТЭС ДЛЯ ТРЁХ СТРАТЕГИЙ')
print("=" * 80)
print()
print_powerplant_comparison(
    system,
    system_static,
    timesteps=TIMESTEPS,
    labels=("Uniform", "Static-tuned"),
)
print()
print_powerplant_comparison(
    system_static,
    system_monthly,
    timesteps=TIMESTEPS,
    labels=("Static-tuned", "Monthly-tuned"),
)

<a id="tbl-comparison-powerplant"> </a>

*Таблица 12: Сравнение статистики расхода силовых установок по трем стратегиям.*

### Сводка прогресса

| Стратегия | Разделение логики | Временное разрешение | Используемая информация |
|:----------------|:----------------|:--------------------|:-----------------|
| **Униформа** | Равные доли всем филиалам | Нет (постоянная) | Нет |
| **Статическая настройка** | Пропорционально шестилетнему общему спросу | Нет (постоянная) | Общий спрос на отрасль |
| **Изменяется ежемесячно** | Сезонные коэффициенты, полученные на основе спроса за календарный месяц | Ежедневное исполнение с повторяющимся сезонным графиком | Ежемесячная потребность в климатологии по отрасли |

На каждом этапе к решению о распределении добавляется дополнительная информация о моделях спроса. Ключевой вывод заключается в том, что даже простейшая осведомленность о спросе (статическая настройка) значительно превосходит равномерное разделение, а ежемесячная вариация обеспечивает дальнейший дополнительный выигрыш. Остается вопрос, на который может ответить только многокритериальная оптимизация (см. [Раздел 12.2](#sec-taqsim-optimisation)), заключается в том, существуют ли лучшие рабочие правила, которые явно уравновешивают одновременно конкурирующие цели. Мы рассмотрим этот вопрос в следующей главе, посвященной оптимизации водораспределения во всей водной системе Зарафшана.

<details style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #3fb618; background-color: #edf7ea;"> 
 <summary> <strong> &#x1F4A1; Совет: Упражнение 2. Чувствительность резервуара </strong> </summary>

Что произойдет с уровнем наполнения Акдарьи, если вы **удвоите** расход попуска по умолчанию?

**Задача:**

1. Создайте базовую систему и запишите расход попуска Акдарьи по умолчанию ( `vr` ).
2. Создать вторую систему, в которой `vr` Акдарьи удваивается.
3. Смоделируйте оба процесса в течение всего ежедневного периода 2017–2022 гг.
4. Сравните окончательный уровень запасов Акдарьи и общий сельскохозяйственный дефицит.

**Подсказка:** После вызова `create_zrb_system()` получите доступ к политике выпуска узла резервуара, чтобы проверить и изменить ее. `ZRBReleaseRule` хранит `vr` как месячный кортеж из 12 элементов в m<sup>3</sup>/s.

**Ожидаемый результат:** Удвоение `vr` приведет к более быстрому истощению запасов Акдарьи, и через несколько лет он достигнет мёртвого объёма. Первоначально это увеличивает подачу воды вниз по течению, но в конечном итоге увеличивает дефицит, поскольку водохранилище не может смягчать сезонную изменчивость. Итоговое хранилище должно быть заметно ниже базового.

</details> 
 <a id="sec-canal-losses"> </a>

### Роль потерь на каналах
Все три базовых сценария включают калиброванные потери при транспортировке воды по каналам по всей транспортной сети, однако общие потери остаются примерно постоянными для разных стратегий и составляют примерно 35% от притока. Эта инвариантность подтверждает, что потери определяются физической инфраструктурой — облицовкой канала, размерами и состоянием, а не правилами распределения. Разделение, пропорциональное спросу, *перераспределяет* потери между ответвлениями (каналы, несущие больше воды, теряют больше), но не уменьшает их общую сумму. Сокращение потерь требует инвестиций в инфраструктуру (облицовку каналов), а не более разумного распределения. Модель потерь и ее калибровка по данным эффективности на уровне системы Узбекистана задокументированы в [Раздел 8](#sec-appendix-calibration).

[Рисунок 13](#fig-loss-breakdown) разлагает общие потери в каналах по механизмам, используя ежемесячно меняющуюся базовую линию. В подавляющем большинстве случаев преобладает фильтрация (>95%), что соответствует преимущественно сети земляных каналов ([Заман и др., 2023 г.](https://doi.org/10.1016/j.agwat.2023.108438); [ФАО, 1989 г.](https://www.fao.org/3/T7202E/T7202E00.htm)). Испарение и эксплуатационные потери незначительны. Правая панель объединяет **ежедневные** результаты моделирования в ежемесячные итоги для удобства чтения, показывая ожидаемые сезонные колебания: пик потерь приходится на лето, когда потоки самые высокие (фильтрация масштабируется с помощью $\sqrt{Q}$ ), а испарение максимальное.

In [ ]:
# @title Роль потерь в каналах · код
fig = plot_canal_loss_breakdown(system_monthly, timesteps=TIMESTEPS)
figures.caption(
    fig,
    'Потери в каналах по типам для сценария с сезонными долями. Фильтрация составляет основную часть потерь; месячные суммы получены из суточного расчёта.',
    label="loss-breakdown",
)

[Рисунок 14](#fig-edge-losses) ранжирует 15 сегментов канала с самыми высокими общими потерями при передаче. Главные магистральные каналы доминируют, поскольку они несут наибольшие объемы потока в сети, даже несмотря на то, что они частично облицованы. Модернизация даже некоторых из этих приоритетных сегментов с полной бетонной облицовкой приведет к непропорциональной экономии воды для всей системы.

In [ ]:
# @title Роль потерь в каналах · код
fig = plot_top_reach_losses(system_monthly, top_n=15)
figures.caption(
    fig,
    '15 участков каналов с наибольшими суммарными потерями при транспортировке воды за суточный расчёт 2017–2022 годов в сценарии с сезонными долями.',
    label="edge-losses",
)

Даже полное устранение всех потерь на каналах не покроет весь дефицит сельского хозяйства. Хотя летние пики спроса и предложения в этом ниво-ледниковом бассейне в целом синхронизированы, система перегружена обязательствами: общие обязательства плюс потери составляют почти 90% притока, а валовые потребности в водозаборе (потребность/эффективность) в отдельных районах могут превышать объемы поставок, достигающие их, даже в месяцы пикового стока. Облицовка каналов и более разумное распределение затрагивают различные аспекты работы системы и являются взаимодополняющими стратегиями.

<details style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #3fb618; background-color: #edf7ea;"> 
 <summary> <strong> &#x1F4A1; Совет: Упражнение 3. Улучшение инфраструктуры </strong> </summary>

Что произойдет с эффективностью системы, если модернизировать **магистральный канал** с земляной на бетонную облицовку?

1. Откройте `data/ZRB_baseline/config/canal_properties.csv`. 
2. Меняем облицовку `Source_to_HW_Ravatkhoza` с `earthen` на `concrete`. 
3. Создайте новую систему с `create_zrb_system(use_canal_losses=True)` и смоделируйте
4. Используйте `compute_water_balance()` для сравнения эффективности новой системы с базовой.

**Ожидаемый результат:** Модернизация магистрального канала должна повысить эффективность системы на 1–3 процентных пункта. Эффект скромный, поскольку фильтрация увеличивается при использовании $\sqrt{Q}$, а не $Q$, поэтому даже каналы с высоким потоком теряют пропорционально меньше на единицу потока.

</details> 
 <details style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #3fb618; background-color: #edf7ea;"> 
 <summary> <strong> &#x1F4A1; Совет: Упражнение 4: Приоритеты облицовки каналов </strong> </summary>

Предположим, у вас есть бюджет на покрытие **100 км** канала бетоном. Используя [Рисунок 14](#fig-edge-losses) и данные о свойствах канала, разработайте инвестиционную стратегию:

1. Загрузите свойства каналов с помощью `load_canal_properties()` и определите земляные каналы, отсортированные по общей потере (из [Рисунок 14](#fig-edge-losses)).
2. Выбирайте каналы для прокладки до тех пор, пока общая длина не достигнет 100 км.
3. Для каждого выбранного канала измените в свойствах его облицовку с `earthen` на `concrete`, пересоберите систему и смоделируйте
4. Сравните эффективность новой системы с базовой.

**Подсказка:** Вы не можете изменить `canal_properties.csv` непосредственно в этом упражнении. Вместо этого после вызова `create_zrb_system(use_canal_losses=True)` получите доступ к правилам потерь на рёбрах сети и программно измените их коэффициенты фильтрации. В бетонных каналах используется $\alpha \approx 0.003$, а не $\alpha \approx 0.050$ для земляных каналов.

**Ожидаемый результат:** Прокладка 100 км земляных каналов с наибольшими потерями должна повысить эффективность системы на 3–6 процентных пунктов.

</details> 
 <div style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #d63384; background-color: #fce4f0;"> 
 <strong> &#x2757; Важно: основные выводы: базовое моделирование </strong> <br> <br>

- Равномерное разделение сточных вод путем доставки больших объемов в отделения с низким спросом и голодания районов с высоким спросом.
- Даже грубое понимание спроса (статическая настройка) значительно повышает результативность водообеспечения и снижает избыточную подачу.
- Месячное разделение обеспечивает дальнейшую дополнительную выгоду за счет адаптации к сезонным моделям спроса.
- **Результативность водообеспечения**(ограничение спроса / общий спрос) — самый справедливый показатель;**Эффективность системы** может вводить в заблуждение, поскольку она вознаграждает за избыточную подачу.
- Потери в каналах остаются примерно постоянными для всех стратегий (<sub> 35% притока) — это физическое свойство, а не настраиваемый параметр. На фильтрация приходится >95% потерь в каналах; испарение и эксплуатационные потери незначительны.
- Магистральные магистральные каналы вносят наибольший вклад в абсолютные потери из-за высоких объемов потока, несмотря на то, что они частично облицованы.
- Устранение всех потерь по каналам не покроет всех дефицитов: система имеет чрезмерные обязательства (обязательства + потери ≈ 90% притока), поэтому абсолютный дефицит воды сохраняется даже тогда, когда летние пики предложения и спроса в целом синхронизированы.

</div> 
 <a id="sec-conclusions"> </a>

# Выводы и перспективы
В этом учебнике был поставлен центральный вопрос: **насколько важно правило распределения в точках отклонения?** Ответ, продемонстрированный с помощью трех постепенно усложняющихся стратегий в одной и той же физической системе, таков: *значительно, но с убывающей отдачей*.

## Чего можно достичь с помощью распределения

Прогрессия по трем базовым уровням демонстрирует четкую закономерность. Равномерное разделение — разделение потока поровну независимо от спроса — дает худший результат: оно наводняет филиалы с низким спросом, одновременно вызывая голод в районах с высоким спросом, создавая большие объемы избыточной подачи, которые не приносят никакой пользы. Даже простейшая осведомленность о потребностях (статическая настройка с одним фиксированным соотношением на каждый сплиттер) значительно повышает результативность водообеспечения и сокращает отходы. Месячное разделение дает дополнительные преимущества за счет адаптации коэффициентов к сезонным моделям спроса, но улучшение от статического к ежемесячному меньше, чем от равномерного к статическому. Урок заключается в том, что **грубая информация о спросе важнее, чем временная точность**: знать, *какие* отрасли нуждаются в воде, важнее, чем знать *точно, когда*.

## Чего не может достичь распределение

Все три стратегии работают в одних и тех же физических ограничениях. Потери при транспортировке воды по каналу, в которых преобладает фильтрация в преимущественно земляной сети каналов, составляют примерно 35% общего притока независимо от того, как распределяется вода. Дробление, пропорциональное спросу, *перераспределяет* потери по отраслям, но не уменьшает их общую сумму. Хотя ниво-ледниковый режим Зерафшана в целом синхронизирует летние пики поставок с пиковым спросом на орошение, система остается чрезмерно загруженной: общие обязательства (</sub> 53% притока) плюс потери в канале (<sub> 35%) требуют почти 90% доступной воды, а потери сами по себе самые высокие летом, потому что фильтрация масштабируется с $\sqrt{Q}$. Никакая эвристика расщепления не сможет устранить этот разрыв. Эти ограничения устраняются двумя взаимодополняющими мерами: **инвестиции в инфраструктуру**(облицовка каналов для уменьшения фильтрации) и**оптимизация политики эксплуатации водохранилищ** (правила попусков, которые перераспределяют воду из периодов высокого расхода в месяцы, когда спрос превышает предложение).

## От моделирования к оптимизации

Эвристические стратегии, рассмотренные здесь, используют информацию о спросе, но не обеспечивают явного баланса конкурирующих целей. Стратегия, меняющаяся ежемесячно, направлена ​​на то, чтобы обеспечить соответствие спроса и предложения повсюду одновременно, но что, если улучшение поставок в одном районе потребует ухудшения показателей в другом? Что, если сокращение сельскохозяйственного дефицита будет происходить за счет надежности охлаждающей воды электростанций? По своей сути это **многоцелевые компромиссы**, по которым эвристика не может систематически ориентироваться.

Учебник по сопутствующей оптимизации посвящен именно этой проблеме. Используя тот же пакет `zarafshan_taqsim` и структуру TaqSim (см. [Раздел 9](#sec-appendix-taqsim)), он применяет **NSGA-II**(многоцелевой генетический алгоритм; см. [Раздел 12.2](#sec-taqsim-optimisation)) для одновременного поиска в пространстве возможных коэффициентов разделения и правил выпуска из резервуара. В результате получается**фронт Парето** — набор решений, в которых ни одна цель не может быть улучшена без ухудшения другой, — что раскрывает истинную поверхность компромисса между конкурирующими целями. Понятия из этого учебника, которые можно перенести непосредственно в учебник по оптимизации, включают:

- Система **водного баланса** для оценки любого возможного решения.
- **Результативность водообеспечения**и**доля избыточной подачи** как показатели производительности.
- Понимание того, что **потери в канале являются физическим ограничением**, а не настраиваемым параметром.
- Роль **политики эксплуатации водохранилищ** в смягчении сезонной изменчивости.
- **сквозное** поведение узлов водопотребления, которое создает взаимозависимости на уровне сети.

<a id="sec-limitations"> </a>

## Допущения и ограничения модели
Прежде чем использовать эту модель для принятия решений, важно понять, чего она не может охватить:

1. **Суточный временной интервал, но все же упрощенные операции.** Модель учитывает ежедневные изменения, но по-прежнему не может фиксировать внутридневные пики паводков, оросительную ротацию или решения оператора, принимаемые с субдневной частотой. Таким образом, буферность водохранилищ и кратковременный дефицит еще можно сгладить относительно реальности.
2. **Статический ряд спроса.** Потребности в орошении представляют собой фиксированные исторические временные ряды. На самом деле фермеры корректируют решения о посадке на основе прогнозов доступности воды. Модель не может уловить эту петлю обратной связи.
3. **Отсутствует взаимодействие с грунтовыми водами.** Потери при просачивании выходят из системы навсегда. В действительности, некоторая утечка пополняет неглубокие водоносные горизонты, из которых фермеры могут откачивать воду. Таким образом, модель переоценивает чистое влияние потерь в каналах на доступность воды для сельского хозяйства.
4. **Эмпирические правила потерь.** В потерях при передаче по каналу используются калиброванные коэффициенты (модель фильтрации $\sqrt{Q}$, сезонные скорости испарения, переведенные в ежедневные потери), а не основанные на физике модели фильтрации. Калибровка нацелена на эффективность на уровне системы, а не на отдельных сегментах канала, поэтому оценки потерь по отдельным рёбрам сети имеют более высокую неопределенность, чем совокупные показатели.
5. **Фиксированная топология сети.** Модель не может отражать изменения инфраструктуры в течение периода моделирования. Структура сети остается постоянной на всех 2191 временных шагах.
6. **Допущение о зоне мёртвого объёма.** Оба резервуара используют долю мёртвого объёма в размере 10% от общей емкости. Фактические значения зависят от отметок водозаборной конструкции, которые могут отличаться от значений по умолчанию.
7. **Нет валидации по конкретному бассейну.** Модель откалибрована по статистическим данным эффективности Узбекистана на национальном уровне и не была проверена на основе наблюдаемых данных, специфичных для бассейна Зеравшан (например, измеренные временные ряды запасов воды в водохранилище, наблюдаемые районные поставки или измеренные на местах потери в канале). Поэтому результаты следует интерпретировать как правдоподобное поведение системы, а не как историческую реконструкцию.

Эти ограничения не делают недействительными результаты модели — они определяют область, в которой результаты имеют смысл. Сопутствующий учебник по оптимизации работает с реалистичными базовыми показателями (включая потери в канале), поэтому оптимальные по Парето решения учитывают потери при транспортировке воды по всей сети.

<a id="sec-exercises"> </a>

# Упражнения
Семь упражнений, включенных в предыдущие разделы, собраны здесь для справки. Они переходят от концептуального понимания через количественный анализ к практическому моделированию.

**Концептуальные и аналитические упражнения:**

- **Упражнение A. Нарисуйте минимальную систему водоснабжения** ( [Раздел 2.2](#sec-taqsim) ): спроектируйте четырехузловую систему и вычислите базовую арифметику водного баланса.
- **Упражнение B — Арифметика спроса и предложения** ( [Раздел 4.1](#sec-data-sources) ): Используйте показатель спроса и предложения, чтобы оценить объемы потерь, доли емкости резервуара и определить, сможет ли устранение потерь закрыть все дефициты.
- **Упражнение C. Прогнозирование перед моделированием** ( [Раздел 5.1](#sec-uniform-baseline) ): спрогнозируйте эффекты разделения, пропорционального спросу, прежде чем увидеть результаты.

**Вычислительные упражнения:**

1. **Упражнение 1. Исследование сети** ( [Раздел 3.6](#sec-sinks) ): Перечислите все различные пути от `Source` до `Sink_Navoi`, используя поиск в глубину в конфигурации ребер.
2. **Упражнение 2 — Чувствительность резервуара** ( [Раздел 5.4](#sec-comparison) ): Удвойте норму сброса по умолчанию Акдарьи и сравните влияние на конечный уровень хранилища и общий сельскохозяйственный дефицит.
3. **Упражнение 3 – Улучшение инфраструктуры** ( [Раздел 5.4.2](#sec-canal-losses) ): Модернизация основного магистрального канала с земляной на бетонную облицовку и измерение влияния на эффективность системы.
4. **Упражнение 4 — Приоритеты облицовки канала** ([Раздел 5.4.2](#sec-canal-losses)): Разработайте инвестиционную стратегию для облицовки 100 км канала и измерьте повышение эффективности системы.

Упражнения развиваются от концептуального понимания (A, B) через прогнозирование и рассуждения (C) к исследованию сети (1), чувствительности параметров (2) и анализу инвестиций в инфраструктуру (3, 4). Вместе они приобретают свободное владение как концепциями предметной области, так и средой моделирования TaqSim.

## Вопросы для обсуждения

Эти вопросы предназначены для семинарского или группового обсуждения. Не существует единственного правильного ответа — цель состоит в том, чтобы рассуждать о компромиссах.

1. **Парадокс эффективности.** Постройте гипотетический сценарий, в котором эффективность системы равна 100 %, а результативность водообеспечения равна 0 %. Что это говорит вам об ограничениях эффективности системы как показателя производительности?

2. **Конфликт между верховьями и низовьями.** Предположим, вы консультируете Министерство водного хозяйства Узбекистана. Район, расположенный выше по течению реки Навои, требует более крупных ассигнований. Какая информация вам понадобится для оценки этого запроса и с какими заинтересованными сторонами вы бы проконсультировались? Как модель помогает сформулировать это решение и о чем она *не* может вам рассказать?

3. **Инфраструктура против распределения.** В учебнике показано, что прокладка каналов и настройка распределения затрагивают различные аспекты работы системы. Если бы у вас был фиксированный бюджет, как бы вы решили, сколько потратить на облицовку канала, а не инвестировать в лучший мониторинг спроса с учетом сезонных колебаний распределения? Какие данные вам понадобятся?

## Вопросы для проверки

Проверьте свое понимание ключевых понятий. Ответы можно найти в соответствующих разделах.

1. Каковы основные типы функциональных узлов в модели ZRB и какой тип узла представляет сегменты канального или речного транспорта?
2. В чем разница между узлом источника и узлом спроса в TaqSim?
3. Каковы четыре зоны в правиле управления по уровню наполнения (SLOP) и что контролирует каждая зона?
4. Почему узлы водопотребления называются «сквозными»? Приведите один пример того, как это влияет на сеть.
5. Какая доля общего притока потребляется на нужды орошения? Какая часть идет на межбассейновые трансферты?
6. Объясните, почему потери на фильтрацию в канале остаются примерно постоянными, независимо от того, какая стратегия разделения используется.
7. Определите *результативность водообеспечения* и *долю избыточной подачи*. Почему результативность водообеспечения является более справедливым показателем, чем эффективность системы?
8. Что такое модель фильтрации $\sqrt{Q}$? Почему фильтрация масштабируется при использовании $\sqrt{Q}$, а не $Q$?
9. Назовите два ограничения модели ZRB, которые могут повлиять на надежность ее результатов.
10. В одном предложении суммируйте, чего *может* достичь и чего *нельзя* достичь в системе ZRB.

<a id="sec-appendix-calibration"> </a>

# Приложение A: Методика калибровки потерь в канале
В этом приложении описано, как калибровались коэффициенты потерь в канале, используемые в [Раздел 5.4.2](#sec-canal-losses). Физические свойства каждого канала (длина, ширина, тип облицовки, состояние), используемые в качестве входных данных для модели потерь, являются оценочными значениями, а не данными, измеренными на месте (см. [Раздел 4.2](#sec-canal-properties)). У нас нет полевых измерений фактических потерь каналов в бассейне Зеравшана. Вместо этого мы калибруем модель потерь по опубликованным данным эффективности на уровне системы для ирригационной инфраструктуры Узбекистана, корректируя коэффициенты до тех пор, пока эффективность моделируемой системы не приблизится к национальному эталонному значению примерно 63% ([Ходжиев и Атаджанова, 2022 г.](https://doi.org/10.1051/e3sconf/202336501011)).

**Примечание об определениях эффективности:** Упоминание 63 % из литературы относится к *эффективности транспортировки* — доле воды, отведенной из источника, которая поступает в точки доставки на уровне поля. Метрика модели `system_efficiency` определяется как `total_delivered / total_inflow`, где знаменатель включает весь приток в систему, включая воду, предназначенную для межбассейновых перебросных приемников, которая никогда не предназначалась для достижения ирригационных районов. Эта разница в определениях означает, что калибровка является приблизительной: мы ориентируемся на правильный порядок величины, а не на точное совпадение. Откалиброванная система достигает КПД примерно 65 %, что соответствует эталонному диапазону с учетом более широкого знаменателя.

<a id="sec-loss-model"> </a>

## Модель потерь
Модель ZRB реализует три типа потерь в каналах по всей транспортной сети:

**Фильтрация** использует модель $\sqrt{Q}$, в которой скорость фильтрации масштабируется в зависимости от смоченного периметра трапециевидного канала:

$$\text{seepage}_{\text{rate}} = \alpha \cdot \sqrt{Q} \cdot L$$

где $Q$ — расход ( m<sup>3</sup>/s), $L$ — длина канала (км), а $\alpha$ — эмпирический калибровочный коэффициент (с неявными единицами, включенными в эмпирическую формулировку), который зависит от облицовки канала. Эта формулировка физически подходит для трапециевидных каналов, где смоченный периметр (и, следовательно, площадь фильтрации) масштабируется с помощью $\sqrt{Q}$ ( [Заман и др., 2023 г.](https://doi.org/10.1016/j.agwat.2023.108438) ).

**Потери от испарения** с поверхности воды в канале меняются ежемесячно, при этом сезонные колебания достигают максимума примерно в 210 мм/месяц летом, согласно данным об испарении в Хорезме (Шибуо и Джаршо, 2007).

**Эксплуатационные** потери от разливов и утечек на гидротехнических сооружениях моделируются как небольшая фиксированная доля потока, зависящая от состояния канала ([ФАО, 1989 г.](https://www.fao.org/3/T7202E/T7202E00.htm)).

<a id="sec-calibration-targets"> </a>

## Цели калибровки
Мы провели калибровку по опубликованным данным по ирригационной инфраструктуре Узбекистана:

| Метрическая | Цель | Источник |
|:-----------------------|:-----------------------|:-----------------------|
| Эффективность системы | 63% (\$\pm\$5%) |  [Ходжиев и Атаджанова (2022)](https://doi.org/10.1051/e3sconf/202336501011) |
| Общие потери канала | \ </sub> 37% притока |  [Лиллис (2020)](https://eurasianet.org/uzbekistan-where-the-amu-darya-goes-to-die) |
| Доминирование фильтрации | \>95% потерь на каналах |  [ФАО (1989)](https://www.fao.org/3/T7202E/T7202E00.htm) ;  [Заман и др. (2023)](https://doi.org/10.1016/j.agwat.2023.108438) |
| Потери на участке | 1--5% на 10 км (большие каналы) | Литературный консенсус |

У нас нет пространственно разрешенных полевых измерений потерь в каналах в бассейне Зерафшана. Таким образом, калибровка нацелена на эффективность на уровне системы, а не на уровень потерь в каждом канале. Это означает, что потери в отдельных каналах могут быть неточными, но поведение всей системы в целом реалистично.

<a id="sec-calibration-procedure"> </a>

## Процедура калибровки
Калибровка проводилась итеративным методом. Во-первых, первоначальное изменение масштаба уменьшило все коэффициенты фильтрации примерно в 7 раз (получено на основе размерного анализа целевой скорости потерь по сравнению с фактической скоростью потерь в репрезентативных условиях потока). Затем в ходе итерационной корректировки было проведено полное моделирование 2017–2022 годов и рассчитан водный баланс: если эффективность системы падала ниже 60%, коэффициенты фильтрации снижались на 10%; если выше 66%, они были увеличены на 10%. Это повторялось до тех пор, пока эффективность не попадала в целевой диапазон. Доля эксплуатационных потерь была снижена в 10 раз, чтобы соответствовать литературным ожиданиям о том, что эксплуатационные потери незначительны по сравнению с утечкой. Наконец, валидация подтвердила, что характер сезонных потерь соответствует сезонным тенденциям испарения, что распределение потерь по типам каналов качественно реалистично (земляной \> частичный \> бетонный) и что ни одно отдельное ребро не теряет больше воды, чем несет.

<a id="sec-calibrated-coefficients"> </a>

## Калиброванные коэффициенты
В следующих таблицах показаны исходные и калиброванные значения коэффициентов:

| Тип подкладки | Начальный $\alpha$ | Калиброванный $\alpha$ | Коэффициент уменьшения |
|:------------|:----------------:|:-------------------:|:----------------:|
| Земляной |      0,350 |        0,050 |        7x |
| Частичный |      0,150 |        0,020 |       7,5x |
| Бетон |      0,020 |        0,003 |       6,7x |

| Состояние канала | Первоначальные операционные потери | Калиброванный |
|:----------------|:------------------------:|:----------:|
| Хорошо |           2,0% |    0,2% |
| Средний |           5,0% |    0,5% |
| Бедные |          12,0% |    1,2% |

Калиброванная система достигает эффективности системы примерно 65 %, при этом общие потери в канале составляют 34,7 % от притока: на фильтрацию приходится 33,1 %, на испарение — 0,2 %, а эксплуатационные потери — на 1,4 %.

<a id="sec-calibration-limitations"> </a>

## Ограничения
Этот подход к калибровке имеет важные ограничения. Без измерений потерь с пространственным разрешением мы не можем подтвердить, что отдельные участки каналов имеют реалистичные темпы потерь. Все каналы одного и того же типа облицовки имеют одинаковый коэффициент фильтрации (допущение единообразного масштабирования). Оценки ширины канала не подтверждены — они были рассчитаны на основе расчетных расходов, а не на основе исследований. Коэффициенты потерь не меняются в зависимости от сезона или возраста канала. Наконец, целевой показатель в 63% является средним показателем по стране для Узбекистана; бассейн Зерафшана может отличаться, но в отсутствие данных по конкретному бассейну лучшим доступным ориентиром являются национальные значения.

<a id="sec-appendix-taqsim"> </a>

# Приложение B: Платформа моделирования TaqSim
 [Раздел 2.2](#sec-taqsim) представил TaqSim на концептуальном уровне, достаточном для того, чтобы следить за моделированием, представленным в учебнике, и интерпретировать его результаты. В этом приложении дается **ориентированный на обучение обзор** того, как организована текущая структура: как моделирование развивается во времени, как разные типы узлов играют разные роли, как события и трассировки поддерживают анализ и как операционные политики превращаются в параметры оптимизации. Он намеренно упрощает некоторые детали API более низкого уровня, поэтому основное внимание уделяется идеям моделирования, а не деталям внутренней реализации.

Подразделы расположены в порядке возрастания абстракции: цикл моделирования ([Раздел 9.1](#sec-sim-loop)) показывает, как вода движется по сети; типы узла и транспорта ( [Раздел 9.2](#sec-taqsim-nodes) и [Раздел 9.3](#sec-taqsim-edges) ) описывают, что происходит в каждом компоненте; события и трассировки ( [Раздел 9.4](#sec-taqsim-events) ) объясняют, как записывается и запрашивается состояние; стратегии и воздействие параметров ( [Раздел 9.5](#sec-taqsim-strategies) и [Раздел 9.6](#sec-taqsim-params) ) показывают, как операционные политики становятся оптимизируемыми параметрами; а цели и оптимизация ( [Раздел 12.1](#sec-taqsim-objectives) и [Раздел 12.2](#sec-taqsim-optimisation) ) соединяют результаты моделирования с анализом компромиссов.

<a id="sec-sim-loop"> </a>

## Цикл моделирования
 `WaterSystem` — оркестратор TaqSim. С точки зрения обучения, он координирует четыре уровня:

| Слой | Ответственность |
|:-----------------------------|:-----------------------------------------|
| **Узел** | Принимает решение или обрабатывает поступившую воду и записывает события. |
| **Участок Reach / транспортный сегмент** | Перемещает воду по каналу или сегменту реки, применяя маршрутизацию, ограничения пропускной способности и физические потери. |
| **Край** | Кодирует, кто с кем подключен в ориентированном графе. |
| **Система водоснабжения** | Проверяет график, строит порядок выполнения и продвигает моделирование на один временной шаг за раз. |

Прежде чем начать моделирование, `WaterSystem.validate()` проверяет, что настроенный граф физически разумен: ссылки на узлы должны быть действительными, граф должен быть ациклическим, источники не могут получать воду вверх по течению, приемники не могут сбрасывать воду дальше, и каждый узел должен в конечном итоге подключиться к розетке. Другими словами, проверка гарантирует, что водохозяйственную систему можно моделировать как сеть с односторонним потоком.

На уровне обучения цикл моделирования можно прочитать так:

```         
for timestep in simulation_period:
    for node in topological_order:
        update node behavior for the current timestep
        route resulting outflows through the configured downstream path
        record all generated events
```

На каждом временном шаге узлы обрабатываются в **топологическом порядке** — порядке, который гарантирует, что каждый компонент посещается только после его вышестоящих предшественников. Это означает, что узел действует только после того, как вода, поступающая за этот временной интервал, уже прибыла.

На практике некоторые узлы генерируют единый отток, тогда как сплиттеры распределяют воду между несколькими ветвями. По мере того как вода движется по настроенному нисходящему пути, транспортные сегменты применяют поведение маршрутизации, пропускной способности и потерь, прежде чем оставшаяся вода достигнет следующего функционального узла.

<details style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #2780e3; background-color: #eaf3fc;"> 
 <summary> <strong> &#x1F4DD; Примечание. Узлам не обязательно знать всю сеть </strong> </summary>

Полезный принцип проектирования заключается в том, что узлам не нужна полная карта бассейна. Им нужно только знать, как вести себя, когда к ним попадает вода. Логика маршрутизации на уровне системы — какие компоненты подключены и в каком порядке они обрабатываются — обрабатывается конфигурацией графа и оркестратором `WaterSystem`.

</details> 
В модели ZRB этот цикл выполняется 2191 раз за ежедневный период моделирования 2017–2022 годов.

<a id="sec-taqsim-nodes"> </a>

## Типы узлов
В учебных целях это помогает различать **функциональные узлы**, которые принимают или принимают решения по управлению водными ресурсами, и **транспортные узлы**, которые представляют физическое движение воды между этими точками принятия решений.

| Тип узла | Преподавательская роль | Пример ZRB |
|:----------|:--------------|:------------|
| Источник | Впрыскивает внешнюю воду в модель | Основной речной приток, поступления местного стока |
| Хранение | Буферизирует воду с течением времени | Каттакурган, Акдарья |
| Сплиттер | Делит доступный поток между конкурирующими отраслями | Водозаборные сооружения HydroWorks |
| Спрос | Забирает воду для использования и регистрирует обслуживание или дефицит | Ирригационные районы |
| Проход | Направляет воду, соблюдая простое ограничение сквозного потока | Навоийская ТЭЦ |
| Выходной узел | Поглощает воду, покидающую моделируемую область | Навои, Джизак, Кашкадарья |
| Достичь | Представляет собой участок канала или реки, по которому транспортируется вода и могут возникнуть потери | Канальный и речной транспортные участки между функциональными узлами |

**Источник.** Источник впрыскивает в систему экзогенную воду. В модели ZRB основным источником является речной приток Раватхозы, дополненный местными источниками стока, связанными с ирригационными районами.

**Хранение.** Узел хранения буферизует воду в зависимости от временных интервалов. Он может накапливать приток, терять воду в результате физических процессов и сбрасывать воду в соответствии с эксплуатационной политикой сброса. Именно так модель представляет способность водохранилищ переносить воду из одной части сезона в другую.

**Делитель.** Делитель распределяет поступающую воду между несколькими ответвлениями, расположенными ниже по течению. В этом учебнике разделители являются ключевыми моментами принятия решений, поскольку различные политики разделения перераспределяют воду между ирригационными районами, стоками и ветвями электростанций.

**Спрос.** Узел спроса представляет пользователя, которому требуется вода. Он забирает воду, регистрирует полезное использование и неудовлетворенный спрос, а также может направлять излишки воды дальше, если поступает больше воды, чем необходимо немедленно.

**PassThrough.** Проходной узел направляет воду вниз по течению, устанавливая при этом простые эксплуатационные условия, такие как ограничение производительности или интерпретация минимального сквозного расхода. В модели ZRB силовая установка представлена ​​именно так, поскольку главный вопрос заключается в том, достаточно ли до нее поступает охлаждающей воды.

**Выходной узел.** Выходной узел — это оконечная розетка. Достигшая его вода покидает моделируемую область и больше не перераспределяется вверх по течению.

**Охват.** Охват представляет собой физический транспортный сегмент между функциональными узлами. Здесь модель может применять эффекты задержки, ограничения пропускной способности и потери при транспортировке, такие как фильтрация или испарение. На многих рисунках в учебниках узлы Reach скрыты для удобства чтения, но они по-прежнему имеют решающее значение для физического реализма модели и диагностики потерь в каналах.

<a id="sec-taqsim-edges"> </a>

## Участки Reach, рёбра и транспортировка воды
В текущей модели обучения полезно отделить **связность графа**от**физического транспорта**:

- **ребро** указывает, какие компоненты связаны в ориентированном графе,
- **участок** обозначает участок канала или реки, по которому фактически течет вода.

Это различие имеет значение, поскольку распределение воды и транспортировка воды – это не одно и то же. Разделитель может решить, куда должна идти вода, но промежуточный транспортный сегмент определяет, сколько воды фактически прибудет после применения ограничений маршрутизации и физических потерь.

Таким образом, транспортную последовательность можно понимать как:

1. вышестоящий узел выпускает или распределяет воду,
2. система направляет воду по настроенному нисходящему соединению,
3. любой промежуточный охват применяет маршрутизацию, емкость и поведение с физическими потерями,
4. оставшаяся вода поступает в нижний узел и регистрируется там.

Благодаря такому разделению **оперативные решения**отличаются от**поведения инфраструктуры**. Это также упрощает расширение модели: можно изменить политику выпуска или разделения, не меняя представления физических потерь, и наоборот.

<div style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #2780e3; background-color: #eaf3fc;"> 
 <strong> &#x1F4DD; Примечание. Модели физических потерь не являются политиками </strong> <br> <br>

Потери, такие как фильтрация и испарение, являются свойствами транспортной инфраструктуры, а не выбором оператора. В модели ZRB представления потерь в канале калибруются один раз (см. [Раздел 8](#sec-appendix-calibration)), а затем фиксируются во всех сценариях. Изменение политики распределения в основном меняет **место** возникновения потерь, а не тот факт, что транспортные потери встроены в инфраструктуру.

</div> 
Для удобства чтения некоторые сетевые рисунки в учебниках не включают узлы охвата. Однако результаты потерь каналов обусловлены этими транспортными сегментами.

<a id="sec-taqsim-events"> </a>

## Система событий и трассировки
TaqSim основан на **событиях**: все состояние системы извлекается из неизменяемого журнала событий, а не из изменяемых переменных состояния. Каждый раз, когда вода генерируется, принимается, хранится, высвобождается, теряется, потребляется, распределяется или выводится, записывается соответствующий класс данных события. Эта архитектура обеспечивает полную апостериорную реконструкцию любого состояния системы на любом временном шаге, сравнение сценариев и построение пользовательских метрик — и все это без повторного запуска моделирования.

### Семейства общих событий

| Семейное мероприятие | Примеры событий | Обучение смыслу |
|:---|:---|:---|
| **Приток и маршрутизация** |  `WaterGenerated` , `WaterReceived` , `WaterDistributed` , `WaterOutput` | Как вода попадает в систему и перемещается от одного компонента к другому |
| **Хранение и выпуск** |  `WaterStored` , `WaterReleased` , `WaterSpilled` | Как резервуары и транспортные ограничения буферизуют, выпускают или отбрасывают воду |
| **Использование и обслуживание** |  `WaterConsumed` , `DeficitRecorded` | Какой объем спроса фактически удовлетворен, а какой остается неудовлетворенным |
| **Убытки** |  `WaterLost` | Как вода уходит через фильтрация, испарение, неэффективность, перелив или подобные процессы |

Точное сочетание событий зависит от типа узла, но идея обучения стабильна: каждое важное движение воды оставляет проверяемый след в журнале событий.

### Запрос событий

События можно запросить несколькими способами:

- `node.events_of_type(WaterLost)` --- все события потерь на всех временных интервалах.
- `node.events_at(t)` --- все события за определенный временной интервал.
- `node.trace(WaterReleased, "amount")` --- извлечь временной ряд сумм выпуска

### Следы

**Trace** — это неизменяемая серия с индексацией по времени ( `dict[int, float]` ), которая предоставляет богатый интерфейс для работы с данными, полученными из событий:

**Строительство:**

- `node.trace(EventType, field)` --- выдержка из журнала событий
- `Trace.from_events(events, field)` --- из списка событий
- `Trace.from_dict({0: 100.0, 1: 150.0})` --- из необработанных данных
- `Trace.constant(50.0, range(TIMESTEPS))` --- единообразное значение во всех временных интервалах.

**Арифметика:** Трассировки поддерживают поэлементные операции как с другими трассами, так и со скалярами: `trace_a + trace_b` , `trace * 0.5` , `trace_a - trace_b` , `trace / scalar` . Бинарные операции используют семантику пересечения — в результате появляются только временные шаги, присутствующие в обеих трассах.

**Преобразование и агрегирование:**

| Метод | Описание |
|:-----------------------------|:-----------------------------------------|
|  `trace.map(fn)` | Применить функцию к каждому значению |
|  `trace.filter(predicate)` | Оставлять только временные шаги, соответствующие условию |
|  `trace.cumsum(initial)` | Текущая накопительная сумма |
|  `trace.sum()` | Итого по всем временным интервалам |
|  `trace.mean()` | Среднее значение |
|  `trace.max()` , `trace.min()` | Крайности |
|  `trace.get(t, default)` | Значение на временном шаге `t` со значением по умолчанию |

В учебнике ZRB `compute_water_balance()` в `analysis.py` использует запросы событий для восстановления полного баланса массы из журнала событий. Метод `trace()` обеспечивает визуализацию временных рядов во всем учебнике: траектории хранения в резервуарах, ежедневные закономерности дефицита и кривые совокупных потерь — все это извлекается из объектов Trace.

<details style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #3fb618; background-color: #edf7ea;"> 
 <summary> <strong> &#x1F4A1; Совет: арифметика трассировки использует семантику пересечений </strong> </summary>

При объединении двух трасс с помощью `+` , `-` , `*` или `/` в результате появляются только **общие временные шаги**. Если одна трасса покрывает всю симуляцию, а другая — только более короткое окно, результат будет ограничен перекрытием. Используйте `.get(t, 0.0)`, если вам нужны значения по умолчанию для отсутствующих временных шагов, или `.reindex(timesteps, fill_value)`, чтобы расширить трассировку до определенного набора временных шагов.

</details> 
 <a id="sec-taqsim-strategies"> </a>

## Стратегии: эксплуатационные правила и физические модели
TaqSim проводит фундаментальное различие между **оперативной политикой** — решениями о том, как управлять системой, — и **физическими моделями** — описанием поведения системы. Это различие определяет, что оптимизатор может и не может изменить.

| Категория | Типовой объект | Оптимизируемый | Цель | Примеры ZRB |
|:-----------|:-----------|:----------:|:-----------|:-----------|
| Операционная политика | Политика выпуска | да | Решите, как резервуары будут сбрасывать воду с течением времени |  `ZRBReleaseRule` (СЛОП) |
| Операционная политика | Политика разделения | да | Решите, как сплиттеры делят поток между филиалами |  `MonthlyDistribution` |
| Физическая модель | Модель потерь при хранении | нет | Представляют испарение или фильтрация хранимой воды | Потери от испарения/фильтрации из резервуара |
| Физическая модель | Модель маршрутизации или потерь | нет | Представление задержки, пропускной способности, фильтрации и испарения на транспортных сегментах |  `Lag`, `CanalLossRule`, `EvaporationLossRule` |

### Базовый класс стратегии

Все операционные политики наследуются от `Strategy`, который предоставляет механизм, делающий возможной оптимизацию. Для целей обучения важными идеями являются:

| Концепция стратегии | Цель |
|:----------------------------------|:-------------------|
|  `__params__` | Называет настраиваемые значения стратегии |
|  `__bounds__` | Ограничивает допустимый диапазон поиска для этих значений |
|  `__constraints__` | Обеспечивает соблюдение таких связей, как суммирование коэффициентов до 1 |
|  `__time_varying__` | Отмечает блоки параметров, которые изменяются в зависимости от сезонного цикла или временных интервалов |
|  `__cyclical__` | Отмечает изменяющиеся во времени параметры, которые повторяются в цикле |

Другими словами, стратегия не просто сообщает модели, как действовать; он также сообщает оптимизатору, **каким частям разрешено перемещаться**и**что считается допустимой конфигурацией**.

### Обнаружение и проверка

Инструменты, ориентированные на оптимизатор, обходят всю систему водоснабжения и собирают только **оперативные стратегии**, привязанные к узлам. Модели физических процессов, такие как модели поглощения пласта или модели потерь на расстоянии, намеренно исключены, поскольку они отражают поведение инфраструктуры, а не выбор оператора.

Затем проверка гарантирует, что наборы параметров-кандидатов остаются значимыми: значения должны оставаться в пределах границ, а такие связи, как упорядоченные пороги хранения или коэффициенты разделения, суммируемые до единицы, должны по-прежнему сохраняться.

### Ограничения

Ограничения выражают отношения между параметрами, которые должны сохраняться после операций генетического алгоритма (кроссовер, мутация). Два особенно важных обучающих примера:

- `SumToOne(params)` --- сумма именованных параметров должна составлять целевое значение (по умолчанию 1,0). Важно для коэффициентов разделения: если у разделителя есть три нижестоящие ветви, их коэффициенты распределения должны быть в сумме равны 1,0.
- `Ordered(low, high)` --- один параметр должен быть меньше или равен другому. Полезно для границ зон SLOP, где неиспользуемое хранилище \< уровень буфера \< уровень сохранения \< емкость.

В текущей модели ZRB `MonthlyDistribution` сохраняет свой график соотношений непосредственно в объекте стратегии, а построитель сети предоставляет повторяющийся сезонный шаблон, используемый при ежедневном моделировании. Напротив, `ZRBReleaseRule` по-прежнему использует настоящие месячно-циклические кортежи для параметров сезонного выпуска. Это полезное напоминание о том, что «изменение во времени» может быть реализовано более чем одним способом, даже в пределах одного и того же семейства моделей.

<a id="sec-taqsim-params"> </a>

## Доступ к параметрам для оптимизации
Генетические алгоритмы работают с плоскими векторами чисел с плавающей запятой, но параметры TaqSim распределяются по вложенным структурам, таким как `system.nodes["dam"].release_policy.vr`. Система **выставления параметров** устраняет этот пробел, предоставляя стандартизированный интерфейс для обнаружения, выравнивания и восстановления параметров.

### Спецификация параметров

Каждый настраиваемый параметр представлен `ParamSpec(path, value)`, где `path` — адрес, разделенный точкой (например, `"HW_Ravatkhoza.split_policy.ratio_HW_AkKaraDarya"`), а `value` — текущее скалярное значение.

### API векторизации

| Метод | Возврат | Цель |
|:---------------------|:------------------------|:------------------------|
|  `param_schema()` |  `list[ParamSpec]` | Откройте для себя все настраиваемые параметры с их путями и текущими значениями |
|  `to_vector()` |  `list[float]` | Сведение параметров к вектору, удобному для оптимизатора |
|  `with_vector(vec)` |  `WaterSystem` | Создайте **новую** систему с параметрами из вектора (неизменяемого) |
|  `param_bounds()` |  `dict[str, tuple]` | Границы на путь параметра |
|  `bounds_vector()` |  `list[tuple]` | Границы, соответствующие порядку `to_vector()` |
|  `constraint_specs()` |  `list[ConstraintSpec]` | Разрешенные ограничения для функции восстановления |
|  `reset()` |  `None` | Очистить события и сбросить состояние для новой симуляции |

Типичная настройка оптимизации:

``` python
# Discover the parameter space
schema = system.param_schema()
for spec in schema:
    print(f"{spec.path} = {spec.value}")

# Flatten to vector
vector = system.to_vector()       # e.g., [0.6, 0.2, 0.2, ...]
bounds = system.bounds_vector()    # e.g., [(0.0, 1.0), (0.0, 1.0), ...]

# Reconstruct from a candidate vector (original unchanged)
candidate = system.with_vector(new_vector)
candidate.simulate(TIMESTEPS)
```

### Расширение изменяющихся во времени параметров

Если параметр стратегии изменяется во времени (например, сезонный блок коэффициентов выпуска или индексированный график распределения), `param_schema()` расширяет его до индексированных записей. Вектор объединяет их в последовательные элементы, а `with_vector()` реконструирует блок структурированных параметров. Границы реплицируются для каждого индекса.

### Исправление ограничений

После скрещивания и мутации вектор-кандидат может нарушать границы или ограничения (например, сумма коэффициентов разделения больше не равна 1,0).  `make_repair(system)` создает функцию восстановления, которая:

1. Обрезает каждое значение по границам его параметра.
2. Применяет исправление ограничений (например, перенормирует группы SumToOne).
3. Возвращает действительный вектор.

Эта функция восстановления применяется к каждому кандидату перед оценкой, гарантируя, что оптимизатор моделирует только физически значимые конфигурации.

<a id="sec-taqsim-objectives"> </a>

## Цели: оценка работы системы
**Objective** оборачивает функцию, которая оценивает одно измерение работы системы после моделирования:

``` python
Objective(
    name="agricultural_deficit",
    direction="minimize",
    evaluate=lambda system: sum_deficit(system, "Dargom"),
    priority=1,
)
```

`direction` указывает, должен ли оптимизатор минимизировать или максимизировать значение. Функция `evaluate` принимает смоделированное значение `WaterSystem` и возвращает одно число с плавающей запятой.

### Встроенные цели

TaqSim предоставляет гибкий API реестра для общих целей:

- `minimize.spill(node_id)` --- общий объем, пролитый на узле Storage или PassThrough.
- `minimize.deficit(node_id)` --- общий неудовлетворенный спрос на узле водопотребления.

Они охватывают наиболее распространенные цели водохозяйственной системы: сокращение отходов (разливов) и улучшение обслуживания (дефицит). Пользовательские цели могут оценивать любую функцию журнала событий, например выработку гидроэлектроэнергии, полученную на основе следов потока, соответствие экологическому потоку или надежность хранения резервуара.

### Декоратор `lift`

Декоратор `lift` преобразует скалярные функции для работы с объектами Trace, позволяя создавать объекты на основе физики. Например, цель гидроэнергетики может поднять степенную функцию ( `P = ρgQh` ) для работы с трассами расхода и напора, а затем агрегировать результат.

### Приложение ZRB

В сопутствующем учебнике по оптимизации используется `minimize.deficit()` для каждого ирригационного района и `minimize.spill()` для двух водохранилищ. Оптимизатор одновременно ищет коэффициенты разделения и параметры выпуска, которые наилучшим образом уравновешивают эти конкурирующие цели.

<a id="sec-taqsim-optimisation"> </a>

## Многоцелевая оптимизация
Стратегии, рассматриваемые в этом учебнике — равномерное, статическое, пропорциональное спросу и ежемесячно меняющееся разделение — являются эвристическими. Они используют знания предметной области для создания разумных правил работы, но не могут найти компромисс между конкурирующими целями. Что, если улучшение доставки Даргома ухудшит качество Нарпая? Что, если сокращение сельскохозяйственного дефицита приведет к увеличению разливов воды из водохранилищ? По сути, это **многоцелевые** проблемы.

### Доминирование и фронт Парето

Решение A **доминирует**над решением B, если A по крайней мере так же хорошо по каждой цели и строго лучше по крайней мере по одной. Решение, в котором не доминирует какое-либо другое решение, называется**оптимальным по Парето**(или недоминируемым). Совокупность всех оптимальных по Парето решений образует**фронт Парето** — границу достижимой производительности:

```
      ↑ перелив (минимизировать)
      │
      │  ╭──────╮  ← фронт Парето
      │ ╱        ╲    (недоминируемые решения)
 high │●          ╲
      │            ●
      │             ╲
  low │              ●
      │
      └────────────────→ дефицит (минимизировать)
           low      high

      НА фронте: оптимальные компромиссы (улучшение требует уступки)
      ВНУТРИ: доминируемые решения (можно улучшить все критерии)
      СНАРУЖИ: недостижимые решения (за пределами возможностей системы)
```

Форма фронта показывает цену компромиссов. **Крутые участки**означают значительное сокращение разливов при небольшом увеличении дефицита;**плоские участки**означают значительное сокращение дефицита при небольшом увеличении разливов.**Точки перегиба** — места, где фронт резко изгибается, — часто представляют собой привлекательные компромиссные решения.

### NSGA-II через ctrl-freak

Механизм оптимизации — **ctrl-freak**( [Лазаро и Зигфрид, 2025 г.](https://github.com/hydrosolutions/ctrl-freak) ), среда генетических алгоритмов, реализованная на NumPy, которая реализует**NSGA-II** (генетический алгоритм с недоминирующей сортировкой II) с подключаемыми стратегиями отбора и выживания. ctrl-freak предназначен только для алгоритмов: он ничего не знает о системах водоснабжения. Механизм предоставления параметров TaqSim ([Раздел 9.6](#sec-taqsim-params)) устраняет этот пробел, преобразуя параметры системы в плоские числовые векторы, с которыми работают генетические алгоритмы, и реконструируя настроенные водохозяйственные системы из векторов-кандидатов.

Алгоритм NSGA-II поддерживает совокупность возможных решений и развивает их на протяжении поколений:

1. **Инициализировать** случайную совокупность в пределах параметров.
2. **Оцените** все цели для каждого кандидата (смоделируйте → вычислите цели).
3. **Повторите** для N поколений:
    а.  Отберите родителей через турнирный отбор с учётом плотности решений (для разнообразия предпочтите более низкий рейтинг Парето, а затем более высокую дистанцию скученности)
    б.  Создать потомство посредством скрещивания (SBX) и мутации (полиномиальной); применить исправление для обеспечения соблюдения границ и ограничений
    в.  Объединить родителей и потомство; ранжировать по недоминируемой сортировке
    д.  Выберите выживших, чтобы заполнить следующее поколение
4. **Верните** `NSGA2Result`, содержащий итоговую популяцию решений, ранги Парето и расстояния скученности (показатели разнообразия).

### API `ctrl_freak.nsga2()`

Цикл оптимизации соединяет алгоритм ctrl-freak с моделированием TaqSim с помощью четырех функций, предоставляемых пользователем:

``` python
from ctrl_freak import nsga2

def init(rng):
    """Initialise one candidate: random parameter vector within bounds."""
    return rng.uniform(lower_bounds, upper_bounds)

def evaluate(x):
    """Simulate one candidate and return objective values."""
    candidate = system.with_vector(x)    # reconstruct WaterSystem
    candidate.simulate(TIMESTEPS)         # run the full daily simulation
    return np.array([obj.evaluate(candidate) for obj in objectives])

def crossover(p1, p2):
    """Recombine two parent vectors (e.g., SBX crossover)."""
    return sbx_crossover(p1, p2)

def mutate(x):
    """Perturb a candidate vector, then repair constraints."""
    return repair(polynomial_mutation(x))

result = nsga2(
    init=init,
    evaluate=evaluate,
    crossover=crossover,
    mutate=mutate,
    pop_size=100,
    n_generations=200,
    seed=42,
    n_workers=4,
)
```

Результатом является `NSGA2Result` (замороженный класс данных), содержащий:

- `result.population` --- `Population` с векторами решения ( `.x` , форма $N \times d$ ) и целевыми значениями ( `.objectives` , форма $N \times m$ )
- `result.rank` --- Ранг Парето каждого решения (ранг 0 = недоминируемый фронт)
- `result.crowding_distance` --- показатель разнообразия каждого решения
- `result.pareto_front` --- удобное свойство, которое извлекает подмножество ранга 0 как новый `Population`.

Чтобы проверить конкретное оптимальное по Парето решение:

``` python
front = result.pareto_front          # Population of non-dominated solutions
best_x = front.x[0]                  # decision vector of first front member
system_star = system.with_vector(best_x)
system_star.simulate(TIMESTEPS)
wb_star = compute_water_balance(system_star)
```

Функция `repair`, созданная `make_repair(system)`, гарантирует, что каждый кандидат, оцененный алгоритмом, физически действителен: границы параметров соблюдаются, а структурные ограничения (например, коэффициенты разделения, суммирующиеся до 1,0) восстанавливаются после кроссовера и мутации ( [Раздел 9.6](#sec-taqsim-params) ).

<div style="padding: 0.8em 1.2em; margin: 1em 0; border-left: 4px solid #d63384; background-color: #fce4f0;"> 
 <strong> &#x2757; Важно: не существует единственного лучшего решения </strong> <br> <br>

Фронт Парето не содержит ни одного наилучшего решения. Каждая точка на фронте одинаково «оптимальна» в смысле Парето: улучшение одной цели обязательно ухудшает другую. Выбор окончательного решения требует **человеческого суждения** о приоритетах: операторы, не склонные к риску, могут предпочесть решения с низким дефицитом; планировщики инфраструктуры могли бы согласиться на более высокий дефицит, чтобы уменьшить нагрузку на резервуар. Фронт Парето предоставляет варианты; приоритеты задают заинтересованные стороны.

</div> 
 <a id="sec-module-reference"> </a>

# Приложение C: Справочник по модулю `zarafshan_taqsim`
Пакет `zarafshan_taqsim` обеспечивает полную реализацию водохозяйственной системы ZRB с шестью модулями. Каждый модуль представлен здесь для справки; Разделы учебника, в которых каждый модуль используется впервые, указаны в скобках.

- **`data.py`** загружает наборы исторических исходных данных из каталога `data/ZRB_baseline/` — приток рек, потребность в орошении для шести районов, осадки, испарение, минимальные экологические потоки, свойства каналов и кривые высоты и объема водохранилища. В текущем рабочем процессе учебника эти исходные ряды преобразуются в ежедневные входные данные модели для периода моделирования 2017–2022 годов. Первичные гидрологические данные (приток рек, потребность в орошении, минимальные расходы) получены из [Рагеттли и др. (2025)](https://doi.org/10.1016/j.ejrh.2025.102185), топология сети взята из Джованоли (2025 г.), а свойства каналов оценены на основе ГИС и литературных источников. (Первое использование: [Раздел 3](#sec-system-model))
- **`strategies.py`** реализует правила эксплуатации, специфичные для ZRB: правило управления по уровню наполнения (SLOP) для резервуаров ( `ZRBReleaseRule` ), сезонно меняющиеся коэффициенты распределения для разделителей ( `MonthlyDistribution` ) и модели потерь в канале ( `CanalLossRule` , `EvaporationLossRule` ). Стратегия `MonthlyDistribution` является ключевым механизмом, обеспечивающим сравнение трех стратегий: она сохраняет график соотношений, используемый каждым разделителем: 365 повторяющихся ежедневных записей для статического базового плана или повторяющуюся сезонную закономерность, полученную на основе ежемесячного спроса для ежемесячно меняющегося базового плана. (Первое использование: [Раздел 5](#sec-baseline-simulations))
- **`network.py`** собирает полную ежедневную систему водоснабжения ZRB через `create_zrb_system()`, предоставляет `visualize_network()` для визуализации ориентированного графа и реализует функции пропорционального разделения спроса `apply_static_demand_splitting()` и `apply_demand_proportional_splitting()`, которые вычисляют коэффициенты распределения на основе моделей спроса ниже по течению. (Первое использование: [Раздел 3.1](#sec-network-topology))
- **`analysis.py`** предоставляет класс данных `WaterBalance` и функцию `compute_water_balance()` — центральный инструмент анализа после моделирования. Он восстанавливает полный баланс массы на основе журнала событий TaqSim, отслеживая приток, потребление, потери (фильтрация, испарение, эксплуатация, переполнение), изменения в хранилище и невязку баланса. (Первое использование: [Раздел 5.1](#sec-uniform-baseline))
- **`visualization.py`** содержит функции отчетности и информационных панелей, используемые в учебнике: `print_annual_water_balance()` для форматированных таблиц водного баланса, `plot_system_dashboard()` для четырехпанельных информационных панелей Plotly и утилиты сравнения для параллельной оценки стратегий. (Первое использование: [Раздел 5.1](#sec-uniform-baseline))
- **`objectives.py`** определяет целевые функции, которые оценивают работу системы после моделирования, предназначенные для вызова из обратного вызова `evaluate` ctrl-freak ( [Раздел 12.2](#sec-taqsim-optimisation) ). Ключевые функции включают `total_agricultural_deficit()`, `equity_deficit()`, `reservoir_spillage()`, `total_spillage()`, `sink_min_flow_deficit()`, `passthrough_min_flow_deficit()` и `release_variability()`. Каждый принимает смоделированный `WaterSystem` и возвращает одно число с плавающей запятой, подходящее для минимизации. (Впервые использовано: сопутствующее учебное пособие по оптимизации)

<a id="sec-glossary"> </a>

# Приложение D: Глоссарий
| Термин | Определение |
|:-----------------------|:----------------------------------------------|
| **Невязка баланса** | Остаток в уравнении водного баланса ($\text{Inflow} - \text{Consumed} - \text{Sink outflow} - \text{Losses} - \Delta\text{Storage}$); значение около нуля подтверждает сохранение массы. |
| **Ограничение** | Отношения между параметрами стратегии, которые должны сохраняться после генетических операций (например, коэффициенты распределения должны быть в сумме равны 1,0). Встроенные типы: `SumToOne`, `Ordered`. |
| **Мёртвый объём** | Вода, хранящаяся под водозаборным сооружением водохранилища, которую невозможно сбросить; в этой модели установлен на уровне 10% полного объёма. |
| **Полезная подача** | Вода доставляется в район в объеме (но не превышающем) его потребности на каждом временном этапе. Поставка сверх спроса считается избыточной поставкой, а не полезной подачей. |
| **Результативность водообеспечения** | Полезная подача/общий спрос. Самый справедливый показатель для сравнения стратегий распределения, поскольку он не вознаграждает за избыточную подачу. |
| **Модель потерь на транспортном участке** | Физическая модель, применяемая к транспортным сегментам для представления фильтрации, испарения или связанных с ними потерь при транспортировке. Это не операционная политика и, следовательно, не цель оптимизации. |
| **На основе событий** | Архитектура, в которой состояние системы восстанавливается из журнала неизменяемых событий (например, `WaterReceived`, `WaterLost`), а не отслеживается с помощью изменяемых переменных состояния. |
| **HydroWorks** | Отводные сооружения (смоделированные как узлы-разделители), которые делят речной поток между конкурирующими рукавами, расположенными ниже по течению. |
| **Межбассейновая переброска** | Вода переносится из одного бассейна реки в другой (например, из Заравшана в Джизакскую область в бассейне Сырдарьи). |
| **NSGA-II** | Генетический алгоритм сортировки без доминирования II — многокритериальный эволюционный алгоритм, используемый в сопутствующем учебнике по оптимизации для поиска оптимальных по Парето решений. |
| **Цель** | Именованная функция, оценивающая работу системы по одному измерению (например, общий дефицит) с направлением (минимизировать или максимизировать). См. [Раздел 12.1](#sec-taqsim-objectives). |
| **Избыточная подача** | Вода доставлена ​​в район сверх его потребностей в заданный период времени. Не дает никаких дополнительных преимуществ в этом районе, но может передаваться к нижестоящим узлам. |
| **Предоставление параметров оптимизатору** | Механизм, с помощью которого TaqSim представляет параметры стратегии в виде плоского вектора с плавающей запятой для оптимизации генетического алгоритма. См. [Раздел 9.6](#sec-taqsim-params). |
| **Фронт Парето** | Набор решений, при которых ни одна цель не может быть улучшена без ухудшения хотя бы одной другой цели. Раскрывает поверхность компромисса между конкурирующими целями. |
| **Проходной** | Поведение узлов водопотребления, когда вода, доставленная сверх потребности, продолжает двигаться вниз по течению к последующим узлам, а не теряется. |
| **SLOP** | Правило управления по уровню наполнения — правило, определяющее расход попуска воды из водохранилища на основе четырех зон хранения: мертвой, буферной, консервационной и зоны контроля наводнений. |
| **Стратегия** | Базовый класс для операционных политик (таких как политики выпуска и разделения), параметры которого настраиваются путем оптимизации. Объявляет метаданные, такие как `__params__`, `__bounds__` и `__constraints__`. См. [Раздел 9.5](#sec-taqsim-strategies). |
| **Эффективность системы** | Общий объем доставки до ворот района / общий приток. Может вводить в заблуждение, поскольку учитываются все поставки, включая воду, доставленную туда, где она не нужна. |
| **Топологическая сортировка** | Упорядочение узлов ориентированного графа, при котором каждый узел обрабатывается после всех своих вышестоящих предшественников. Используется TaqSim для определения порядка выполнения моделирования. |
| **Временной ряд Trace** | Неизменяемый ряд с индексацией по времени (шаг времени → значение), извлеченный из журналов событий и поддерживающий арифметические операции, операции преобразования и агрегирования. См. [Раздел 9.4](#sec-taqsim-events). |
| **Доля избыточной подачи** | Избыточная подача / общий приток. Измеряет долю притока системы, превышающую потребности любого округа. |
| **Водный баланс** | Учет массового сохранения, который отслеживает всю воду от притока до потребления, потерь, изменений в хранилищах и конечного оттока. |

# Ссылки

# Литература

1. **Ragettli et al. (2025).**Unraveling Agricultural Water Use in Three {Central Asian. *Journal of Hydrology: Regional Studies*.**58**, 102185. [doi:10.1016/j.ejrh.2025.102185](https://doi.org/10.1016/j.ejrh.2025.102185)
2. **Groll et al. (2015).**Water Quality, Potential Conflicts and Solutions—an Upstream--Downstream Analysis of the Transnational {Zarafshan. *Environmental Earth Sciences*.**73**, 743--763. [doi:10.1007/s12665-013-2988-5](https://doi.org/10.1007/s12665-013-2988-5)
3. **Kulmatov et al. (2013).**Assessment of Water Quality of the Trans-Boundary {Zarafshan. *Journal of Water Resource and Protection*.**5**, 17--30. [doi:10.4236/jwarp.2013.51003](https://doi.org/10.4236/jwarp.2013.51003)
4. **Shultz (1965).** Rivers of {Middle Asia.
5. **CAWater-Info (2024).** Reservoirs in {Uzbekistan. [https://cawater-info.net/bk/1-1-1-1-3-uz_e.htm](https://cawater-info.net/bk/1-1-1-1-3-uz_e.htm)
6. **CAWater-Info (2024).** Water Resources of {Zeravshan. [https://www.cawater-info.net/zeravshan/water_e.htm](https://www.cawater-info.net/zeravshan/water_e.htm)
7. **Rakhmatullaev & Huneau (2011).**Geostatistical Approach for the Assessment of the Water Reservoir Capacity in Arid Regions: A Case Study of the {Akdarya. *Environmental Earth Sciences*.**63**, 447--460. [doi:10.1007/s12665-010-0711-3](https://doi.org/10.1007/s12665-010-0711-3)
8. **Karimov et al. (1995).** Fisheries in the {Zarafshan. [https://www.fao.org/4/v9529e/v9529e06.htm](https://www.fao.org/4/v9529e/v9529e06.htm)
9. **Monitor (2024).** Navoi Power Station. [https://www.gem.wiki/Navoi_power_station](https://www.gem.wiki/Navoi_power_station)
10. **Giovanoli (2025).** Cooling Water Requirements of the {Navoi.
11. **Giovanoli (2025).** Optimising Water Allocation in the {Zarafshan.
12. **USAID (n.d.).** Irrigation Map of the {Zarafshan.
13. **FAO (1989).** Irrigation Water Management: Irrigation Scheduling. [https://www.fao.org/3/T7202E/T7202E00.htm](https://www.fao.org/3/T7202E/T7202E00.htm)
14. **Khodjiyev & Atadjanova (2022).**The Current State of Irrigation Networks and Their Use in the Water Sector of the Republic of {Uzbekistan. *E3S Web of Conferences*.**365**, 01011. [doi:10.1051/e3sconf/202336501011](https://doi.org/10.1051/e3sconf/202336501011)
15. **Zaman et al. (2023).**Characterization and Control of Irrigation Canal Seepage Losses: A Review. *Agricultural Water Management*.**287**, 108438. [doi:10.1016/j.agwat.2023.108438](https://doi.org/10.1016/j.agwat.2023.108438)
16. **Shibuo & Jarsj\"o (2007).** Hydrological Responses to Climate Change and Irrigation in the {Aral Sea.
17. **Lillis (2020).** Uzbekistan: Where the {Amu Darya. *Eurasianet*. [https://eurasianet.org/uzbekistan-where-the-amu-darya-goes-to-die](https://eurasianet.org/uzbekistan-where-the-amu-darya-goes-to-die)
18. **Deb et al. (2002).**A Fast and Elitist Multiobjective Genetic Algorithm: {NSGA-II. *IEEE Transactions on Evolutionary Computation*.**6**, 182--197. [doi:10.1109/4235.996017](https://doi.org/10.1109/4235.996017)
19. **Maier et al. (2014).**Evolutionary Algorithms and Other Metaheuristics in Water Resources: Current Status, Research Challenges and Future Directions. *Environmental Modelling & Software*.**62**, 271--299. [doi:10.1016/j.envsoft.2014.09.013](https://doi.org/10.1016/j.envsoft.2014.09.013)
20. **Reed et al. (2013).**Evolutionary Multiobjective Optimization in Water Resources: The Past, Present, and Future. *Advances in Water Resources*.**51**, 438--456. [doi:10.1016/j.advwatres.2012.01.005](https://doi.org/10.1016/j.advwatres.2012.01.005)
21. **Lazaro & Siegfried (2024).** {TaqSim. [https://github.com/hydrosolutions/taqsim](https://github.com/hydrosolutions/taqsim)
22. **Lazaro & Siegfried (2025).** {ctrl-freak. [https://github.com/hydrosolutions/ctrl-freak](https://github.com/hydrosolutions/ctrl-freak)
23. **Lazaro & Siegfried (2025).** ctrl-freak User Guide. [https://github.com/hydrosolutions/ctrl-freak/blob/b9c20e3/scratchpad/ctrl-freak-guide.md](https://github.com/hydrosolutions/ctrl-freak/blob/b9c20e3/scratchpad/ctrl-freak-guide.md)